# 🚀 StockPulse AI — 股市智慧儀表板 Capstone 實戰 **（ADK 2.0 版）**

## ADK 2.0 `Workflow` × 真實 A2A Protocol × LangGraph 橋接 × FastAPI × Cloud Run

---

## 這一版跟舊版差在哪

舊版教材有三個地方是「講了但沒真的做」，這一版全部換成真的：

| 主題 | 舊版（ADK 1.x 時期） | 這一版（ADK 2.0） |
|---|---|---|
| ADK Agent | `class MockMarketAnalystAgent` — 純 Python 假物件，沒進 ADK | 真的 `LlmAgent` + `FunctionTool`，跑在 `Runner` 上 |
| 多步驟編排 | `SequentialAgent` / `ParallelAgent` | **`Workflow`**（ADK 2.0 起前兩者已 deprecated），真 DAG、可 fan-out/fan-in/條件分支 |
| 看得到流程圖 | ❌ 只有 ASCII art 手畫 | ✅ `plot_workflow_graph()` 直接把 graph 畫出來，還能**用執行狀態上色** |
| A2A | 一個 dict 假裝是 AgentCard + 一個 `async def` 假裝是協定 | ✅ `to_a2a()` 起真的 A2A server、真的 `/.well-known/agent-card.json`、真的 JSON-RPC over HTTP、真的 SSE streaming、`RemoteA2aAgent` 當 client |
| LangGraph | 跟 ADK 各跑各的 | ✅ `LangGraphAgent` 橋接進 ADK，LangGraph 圖變成一個 ADK node |
| 非同步 | `nest_asyncio` + `asyncio.run()` | ✅ 直接 top-level `await`（Python 3.14 + ipykernel 7 已原生支援；**`nest_asyncio` 在這版會弄壞 uvicorn**，見 §5） |

---

## 系統架構

```
┌──────────────────────────────────────────────────────────────────────────┐
│                          📱 Frontend (React / HTML Dashboard)             │
└─────────────────────────────────┬────────────────────────────────────────┘
                                  │ HTTP / REST
┌─────────────────────────────────▼────────────────────────────────────────┐
│                        🌐 FastAPI Gateway  (§6)                           │
│   /api/data/*   /api/analyze/{ticker}   /api/podcast/*                    │
│   /a2a/market-analyst/*        ←─ mount A2A Starlette app                 │
│   /a2a/trading-strategist/*    ←─ mount A2A Starlette app                 │
└─────────────────────────────────┬────────────────────────────────────────┘
                                  │
              ┌───────────────────▼───────────────────┐
              │   🔗 A2A Coordinator Workflow  (§5)    │
              │   ADK 2.0 Workflow，節點是遠端 agent   │
              │                                        │
              │      START                             │
              │        │  fan-out（同時發車）           │
              │   ┌────┴─────┐                         │
              │   ▼          ▼                         │
              │ Remote     Remote                      │
              │ A2A #1     A2A #2                      │
              │   │          │                         │
              │   └────┬─────┘  JoinNode（等兩邊到齊）  │
              │        ▼                               │
              │   synthesiser (LlmAgent)               │
              └────┬───────────────────┬───────────────┘
                   │ JSON-RPC          │ JSON-RPC
                   │ over HTTP         │ over HTTP
   ┌───────────────▼──────────┐  ┌─────▼────────────────────────┐
   │ 🏦 Market Analyst  (§3)   │  │ 📈 Trading Strategist  (§4)  │
   │ ADK 2.0 Workflow          │  │ LangGraph → LangGraphAgent   │
   │ START→parse→fan-out×3     │  │ indicators→strategy→risk     │
   │ →JoinNode→LlmAgent→route  │  │ 用 ADK 橋接包成 ADK agent    │
   └───────────┬───────────────┘  └─────┬────────────────────────┘
               │                        │
               └────────┬───────────────┘
                        ▼
        📊 yfinance（報價 / 歷史 / 新聞）+ 🤖 Gemini API
```

---

## 執行前須知

- **必要**：Google AI Studio API Key（免費額度夠跑完）→ https://aistudio.google.com/apikey
- **Python**：3.11+（本教材在 3.14 驗證）
- **執行時間**：全部跑完約 **6～12 分鐘**，主要花在 Gemini 呼叫與 yfinance 抓資料
- **費用**：全程用 `gemini-*-flash` 系列，約 15 次呼叫，免費額度內
- **網路**：yfinance、Gemini、A2A（localhost）都需要；A2A server 跑在本機背景 thread，不對外開埠
- **圖形**：graph 用 graphviz 畫。macOS `brew install graphviz`／Ubuntu `apt install graphviz`／Colab 已內建

## 目錄

| 章節 | 內容 |
|---|---|
| §0 | 環境、API Key、選模型、中文字型 |
| §1 | 資料層：yfinance + 技術指標（這些會變成 ADK 的 tool） |
| §2 | ADK 2.0 核心：`LlmAgent` / `App` / `Runner` / `Event` |
| §3 | **ADK 2.0 `Workflow`：把 graph 叫出來**（含執行狀態上色） |
| §4 | LangGraph Trading Strategist + `LangGraphAgent` 橋接 |
| §5 | **真正的 A2A Protocol**（AgentCard / JSON-RPC / SSE / Coordinator Workflow） |
| §6 | FastAPI Gateway（mount A2A app） |
| §7 | Dashboard |
| §8 | Podcast 觀點融合（選配） |
| §9 | 收尾：關掉背景 server |
| §10 | Docker + Cloud Run 部署 |
| §11 | ADK 1.x → 2.0 遷移對照表 |
| §12 | 參考資料與檢核清單 |

# 0️⃣ 環境、API Key、模型與字型

這一節做四件事，後面每一節都靠它：

1. 裝套件、印版本（版本對不上時第一時間就看得出來）
2. 拿 API Key —— **用一個「互動能問、headless 不卡」的寫法**
3. 自動挑一個你的帳號真的有權限的 Gemini 模型
4. 把 matplotlib 的中文字型設好，圖表才不會變成一排豆腐方塊

In [ ]:
# ── 安裝套件（第一次執行才需要，把下面那行的 # 拿掉）──────────────────────
# ADK 2.x 要 a2a extra 才會把 a2a-sdk 一起裝進來
# !pip install -q "google-adk[a2a]>=2.8,<3" "langgraph>=1.2" langchain-google-genai \
#     yfinance pandas matplotlib fastapi "uvicorn[standard]" httpx python-dotenv graphviz
#
# 用 uv 的話（本教材推薦）：
# !uv pip install -q "google-adk[a2a]>=2.8,<3" "langgraph>=1.2" langchain-google-genai \
#     yfinance pandas matplotlib fastapi "uvicorn[standard]" httpx python-dotenv graphviz
#
# graphviz 的「dot」執行檔要另外裝（畫 graph 用）：
#   macOS  : brew install graphviz
#   Ubuntu : sudo apt-get install -y graphviz
#   Colab  : 已內建，不用裝

import importlib.metadata as md
import shutil
import sys

print(f"Python           {sys.version.split()[0]}")
for pkg in ["google-adk", "google-genai", "a2a-sdk", "langgraph", "langchain-core",
            "langchain-google-genai", "yfinance", "pandas", "fastapi", "uvicorn", "graphviz"]:
    try:
        print(f"{pkg:<24} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<24} ❌ 未安裝")

dot = shutil.which("dot")
print(f"\ngraphviz `dot` 執行檔  {dot or '❌ 找不到（graph 會退回文字模式，其他功能不受影響）'}")

# 本教材驗證過的版本：google-adk 2.8.0 / google-genai 2.20.0 / a2a-sdk 1.1.2
# / langgraph 1.2.11 / yfinance 1.7.0 / pandas 3.0.5 / Python 3.14.7

In [ ]:
# ── 取得 Google API Key ────────────────────────────────────────────────────
# 為什麼不用 input()？
#   舊版教材寫 api_key = input("請輸入您的 Google API Key: ")
#   這在 JupyterLab 可以，但用 nbclient / papermill / CI 自動跑整本時，
#   input() 會直接丟 StdinNotImplementedError，整本執行就斷在這裡。
# 正確做法：環境變數 → .env → 只有「真的有前端能回答」時才 getpass。
import os
import pathlib


def can_prompt() -> bool:
    """現在的環境有沒有辦法跟人要輸入？

    JupyterLab / Colab → True；nbclient / papermill 這種 headless → False。
    注意 sys.stdin.isatty() 在三種情況都是 False，分不出來，不能用。
    """
    try:
        return bool(get_ipython().kernel._allow_stdin)  # noqa: F821
    except Exception:
        return False


# google-genai 與 ADK 兩個名字都認，所以兩個都要找——只查一個會讓
# 「明明設了 GEMINI_API_KEY 卻說沒有 key」這種假陰性發生。
ACCEPTED_KEY_VARS = ("GOOGLE_API_KEY", "GEMINI_API_KEY")


def load_api_key(dotenv_paths: tuple = (".env", "../.env", "~/.env")) -> tuple:
    """依序嘗試：環境變數 → .env 檔 → 互動輸入。拿不到就回 (None, "missing")，不卡住。"""
    for var in ACCEPTED_KEY_VARS:                    # 1) 環境變數（兩個名字都算）
        if os.environ.get(var):
            return os.environ[var], f"env:{var}"

    for p in dotenv_paths:                           # 2) .env 檔
        path = pathlib.Path(p).expanduser()
        if not path.is_file():
            continue
        try:
            from dotenv import load_dotenv
            load_dotenv(path, override=False)
        except ImportError:                          # 沒裝 python-dotenv 就手動讀
            for line in path.read_text(encoding="utf-8").splitlines():
                if "=" in line and not line.lstrip().startswith("#"):
                    k, _, v = line.partition("=")
                    os.environ.setdefault(k.strip(), v.strip().strip("'\""))
        for var in ACCEPTED_KEY_VARS:
            if os.environ.get(var):
                return os.environ[var], f"dotenv:{path}"

    if can_prompt():                                 # 3) 只在互動環境才問
        import getpass
        v = getpass.getpass("請輸入 GOOGLE_API_KEY（不會顯示）: ").strip()
        if v:
            os.environ["GOOGLE_API_KEY"] = v
            return v, "getpass"

    return None, "missing"                           # 4) headless 缺 key → 不卡住


GOOGLE_API_KEY, KEY_SOURCE = load_api_key()
HAS_KEY = bool(GOOGLE_API_KEY)

if HAS_KEY:
    # 只設 GOOGLE_API_KEY。兩個都設的話 google-genai 會在 stderr 唸一句
    # 「Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.」
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    os.environ.setdefault("GOOGLE_GENAI_USE_VERTEXAI", "0")   # 用 AI Studio 的 key

print(f"Key 來源: {KEY_SOURCE} | 有 key: {HAS_KEY} | 可互動輸入: {can_prompt()}")
if HAS_KEY:
    print(f"✅ API Key 已設定（前 8 碼 {GOOGLE_API_KEY[:8]}…）")
else:
    print("⚠️ 沒有 API Key。資料層 / graph 繪製那些不用 LLM 的 cell 照樣能跑，")
    print("   需要呼叫 Gemini 的 cell 會自動印 SKIPPED 跳過，不會讓整本執行中斷。")

In [ ]:
# ── 自動挑一個「你的帳號真的有權限」的模型 ───────────────────────────────
# 教材寫死 model 名稱是最常見的壞掉原因：模型會下架、會改名、不同帳號權限不同。
# 這裡直接問 API 你有什麼，再按偏好順序挑。
MODEL_PREFERENCE = [
    "gemini-3.5-flash",       # 新、快、便宜
    "gemini-3-flash-preview",
    "gemini-flash-latest",    # 永遠指向當前最新 flash
    "gemini-2.5-flash",       # 最穩定的保底
]

MODEL = None
AVAILABLE_MODELS = []

if HAS_KEY:
    from google import genai

    client = genai.Client(api_key=GOOGLE_API_KEY)
    AVAILABLE_MODELS = sorted(
        m.name.removeprefix("models/")
        for m in client.models.list()
        if "generateContent" in (m.supported_actions or [])
    )
    MODEL = next((m for m in MODEL_PREFERENCE if m in AVAILABLE_MODELS), None)
    if MODEL is None:  # 偏好清單全落空 → 挑任一個 flash
        MODEL = next((m for m in AVAILABLE_MODELS if "flash" in m and "image" not in m
                      and "tts" not in m), None)

    print(f"✅ 選用模型: {MODEL}")
    print(f"   你的帳號共有 {len(AVAILABLE_MODELS)} 個支援 generateContent 的模型")
    print(f"   其中 gemini 系列: {[m for m in AVAILABLE_MODELS if m.startswith('gemini')][:12]}")
else:
    MODEL = "gemini-2.5-flash"   # 沒 key 也給個值，後面 cell 的 import 不會炸
    print(f"⚠️ 沒 key，MODEL 先設為 {MODEL}（不會真的呼叫）")

In [ ]:
# ── 全域設定：安靜的 log + 中文字型 ──────────────────────────────────────
import json
import logging
import warnings

import pandas as pd

# 只關掉「已知的、無資訊量的」噪音，不要用光禿禿的 warnings.filterwarnings("ignore")——
# 那會把整個 kernel 之後所有警告都靜音，包含你真的想看到的那些。
warnings.filterwarnings("ignore", message=r"\[EXPERIMENTAL\].*")      # ADK 的 A2A 模組
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
# AFC 那段長篇建議是用 logging 發的，不是 warnings，所以要關 logger
logging.getLogger("google_genai").setLevel(logging.ERROR)             # 含 .models 子 logger
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google.adk").setLevel(logging.ERROR)
# yfinance 遇到查不到的代碼會把 `HTTP Error 404: {...}` 直接印到 stderr。
# 我們的函式其實已經好好回傳 {"error": ...} 了，但學生看到紅字會以為當掉。
logging.getLogger("yfinance").setLevel(logging.CRITICAL)


# ── matplotlib 中文字型 ──────────────────────────────────────────────────
# 沒設字型的話，「收盤價」「布林帶」會變成 □□□。
def setup_cjk_font() -> str | None:
    """找一個系統上真的有的中文字型設給 matplotlib，找不到就回 None。"""
    import matplotlib
    import matplotlib.font_manager as fm

    candidates = [
        "PingFang TC", "PingFang HK", "PingFang SC", "Heiti TC",        # macOS
        "Apple LiGothic", "Arial Unicode MS", "Hiragino Sans GB",
        "Microsoft JhengHei", "Microsoft YaHei", "SimHei",              # Windows
        "Noto Sans CJK TC", "Noto Sans TC", "Noto Sans CJK SC",         # Linux / Colab
        "WenQuanYi Zen Hei", "Taipei Sans TC Beta",
    ]
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in candidates:
        if name in installed:
            matplotlib.rcParams["font.sans-serif"] = [name] + \
                matplotlib.rcParams["font.sans-serif"]
            matplotlib.rcParams["axes.unicode_minus"] = False   # 負號才不會變方塊
            return name
    return None


CJK_FONT = setup_cjk_font()
# 圖表標籤：有中文字型就用中文，沒有就退回英文（總比一堆豆腐好）
L = (lambda zh, en: zh) if CJK_FONT else (lambda zh, en: en)

print(f"中文字型: {CJK_FONT or '❌ 找不到 → 圖表標籤自動改用英文'}")
if not CJK_FONT:
    print("   想要中文圖表的話：Colab 執行 "
          "`!apt-get install -y fonts-noto-cjk` 後 restart runtime")

# 這本筆記本全程用 top-level await，不用 asyncio.run、不用 nest_asyncio（§5 會解釋為什麼）
import asyncio
print(f"目前 cell 已經在 event loop 裡: {asyncio.get_event_loop_policy() is not None} "
      f"→ 直接用 await 就好")

# 1️⃣ 資料層：yfinance + 技術指標

這一節的四個函式，稍後會**原封不動變成 ADK Agent 的 tool**。所以寫法上有三個約束：

1. **參數要有型別註解**——ADK 靠它自動生出給模型看的 function schema
2. **docstring 要寫清楚**——那就是模型決定「要不要呼叫這個 tool」的依據
3. **回傳值必須是 JSON 可序列化的 dict**——不能塞 DataFrame 進去

> ⚠️ **yfinance 1.7 的兩個坑（舊版教材會踩到）**
> - `.news` 的結構是 `[{"id":…, "content": {…}}]`，欄位藏在 `content` 裡，不是平的
> - `dividendYield` 現在**已經是百分比數字**（`0.46` 就是 0.46%），舊版教材再乘 100 會變成 46%

In [ ]:
import warnings

import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")


def _num(value, digits: int = 2, default=0):
    """安全取數字：None / pd.NA / NaN / inf / 非數字都回 default。

    為什麼需要這個：pandas 3 會回 numpy scalar 或 pd.NA，而 `round(None, 2)` 直接
    丟 TypeError。另外 RSI 的 gain/loss 在「14 天內沒有下跌日」時會變成 inf，
    inf 進 json.dumps 會產生非法 JSON（`Infinity`），LLM 那邊會解析失敗。
    每個數值欄位都過這一關，就不用在四個函式裡各寫一次防護。
    """
    try:
        if value is None:
            return default
        f = float(value)
        if f != f or f in (float("inf"), float("-inf")):     # NaN / ±inf
            return default
        return round(f, digits)
    except (TypeError, ValueError):
        return default


def get_stock_info(ticker: str) -> dict:
    """取得一支股票的即時報價與基本面資訊。

    Args:
        ticker: 股票代碼。美股直接用代號（NVDA、AAPL）；台股要加 .TW（2330.TW）。

    Returns:
        含公司名稱、現價、前收、當日高低、52 週高低、市值、本益比、股利率、成交量的 dict。
        取不到資料時回傳 {"error": "..."}。
    """
    try:
        info = yf.Ticker(ticker).info
        if not info or len(info) <= 1:
            return {"error": f"找不到 {ticker} 的資料，請確認代碼（台股要加 .TW）"}

        return {
            "ticker": ticker,
            "company_name": (info.get("longName") or info.get("shortName")
                             or info.get("displayName") or "N/A"),
            # yfinance 1.7：盤中有 currentPrice，收盤後可能只有 regularMarketPrice
            "current_price": _num(info.get("currentPrice") or info.get("regularMarketPrice")),
            "previous_close": _num(info.get("previousClose")),
            "open": _num(info.get("open")),
            "day_high": _num(info.get("dayHigh")),
            "day_low": _num(info.get("dayLow")),
            "week52_high": _num(info.get("fiftyTwoWeekHigh")),
            "week52_low": _num(info.get("fiftyTwoWeekLow")),
            "market_cap": int(info.get("marketCap") or 0),
            "pe_ratio": _num(info.get("trailingPE")),
            # ⚠️ yfinance 1.7 這個值已經是「百分比」了，不要再 *100
            "dividend_yield_pct": _num(info.get("dividendYield")),
            "currency": info.get("currency", ""),
            "volume": int(info.get("volume") or 0),
            "average_volume": int(info.get("averageVolume") or 0),
        }
    except Exception as e:
        return {"error": f"無法取得 {ticker} 資訊: {e}"}


def get_stock_history(ticker: str, period: str = "3mo") -> dict:
    """取得一段時間的歷史價格摘要（起訖價、漲跌幅、區間高低、均量）。

    Args:
        ticker: 股票代碼，例如 'NVDA' 或 '2330.TW'。
        period: 時間範圍，可用 '1mo'、'3mo'、'6mo'、'1y'、'2y'、'5y'。

    Returns:
        含起訖日期與價格、漲跌金額與百分比、期間高低、平均成交量的 dict。
    """
    try:
        df = yf.Ticker(ticker).history(period=period)
        if df.empty:
            return {"error": f"{ticker} 在 {period} 區間沒有歷史資料"}

        first_close = float(df["Close"].iloc[0])
        last_close = float(df["Close"].iloc[-1])
        change = last_close - first_close
        change_pct = (change / first_close * 100) if first_close else 0.0

        return {
            "ticker": ticker,
            "period": period,
            "data_points": int(len(df)),
            # index 是有時區的 DatetimeIndex，strftime 後才好 JSON 序列化
            "start_date": df.index[0].strftime("%Y-%m-%d"),
            "end_date": df.index[-1].strftime("%Y-%m-%d"),
            "start_price": _num(first_close),
            "end_price": _num(last_close),
            "price_change": _num(change),
            "price_change_pct": _num(change_pct),
            "high_price": _num(df["Close"].max()),
            "low_price": _num(df["Close"].min()),
            "avg_volume": int(_num(df["Volume"].mean(), 0)),
            "trend": "📈 上升" if change > 0 else "📉 下降",
        }
    except Exception as e:
        return {"error": f"無法取得 {ticker} 歷史數據: {e}"}


def get_market_news(ticker: str, top_n: int = 5) -> dict:
    """取得一支股票最近的新聞標題與摘要。

    Args:
        ticker: 股票代碼，例如 'NVDA' 或 '2330.TW'。
        top_n: 最多回傳幾則，預設 5。

    Returns:
        含 news_count 與 items（標題／摘要／來源／連結／時間）的 dict。
    """
    try:
        news = yf.Ticker(ticker).news or []
        items = []
        for entry in news[:top_n]:
            # yfinance 1.7 是 {"id":…, "content": {…}}；舊版是平的 dict。兩種都吃。
            c = entry.get("content") or entry
            provider = c.get("provider") or {}
            canonical = c.get("canonicalUrl") or {}
            items.append({
                "title": c.get("title", ""),
                "summary": (c.get("summary") or c.get("description") or "")[:300],
                "publisher": provider.get("displayName") or entry.get("publisher", ""),
                "link": canonical.get("url") or entry.get("link", ""),
                "published": str(c.get("pubDate") or entry.get("providerPublishTime", "")),
            })
        return {"ticker": ticker, "news_count": len(news), "items": items}
    except Exception as e:
        return {"ticker": ticker, "news_count": 0, "items": [], "error": str(e)}


print("✅ get_stock_info / get_stock_history / get_market_news 已定義")

In [ ]:
# ── 測一下：美股 + 台股，還有一個故意打錯的代碼 ─────────────────────────
for t in ["NVDA", "2330.TW"]:
    info = get_stock_info(t)
    print(f"【{t}】{info.get('company_name')}")
    print(f"   現價 {info.get('current_price')} {info.get('currency')} "
          f"| 前收 {info.get('previous_close')} "
          f"| P/E {info.get('pe_ratio')} "
          f"| 股利率 {info.get('dividend_yield_pct')}%")
    hist = get_stock_history(t, "3mo")
    print(f"   3個月 {hist.get('start_price')} → {hist.get('end_price')} "
          f"({hist.get('price_change_pct')}%) {hist.get('trend')}")
    news = get_market_news(t, top_n=2)
    for n in news["items"]:
        print(f"   📰 [{n['publisher']}] {n['title'][:60]}")
    print()

# 錯誤處理：壞代碼要回 error，不能丟例外（不然 agent 呼叫 tool 時會整個掛掉）
#
# ⚠️ 這裡有個反直覺的地方：yfinance 1.7 對查不到的代碼**不會丟例外**，
#    `yf.Ticker("NOTATICKERXYZ").info` 會回一個只有 1 個 key 的 dict
#    （`{'trailingPegRatio': None}`）。所以只寫 try/except 是抓不到的，
#    要靠「資料少得不合理」來判斷 —— 這就是 get_stock_info 裡 `len(info) <= 1` 那一行。
#    （空字串代碼倒是會丟 ValueError，try/except 有用。）
#    另外 yfinance 會把 404 印到 stderr，我們在 §0 把它的 logger 關掉了，
#    不然這裡會出現一片紅字，看起來像當掉。
print("打錯代碼的行為 →", get_stock_info("NOTATICKERXYZ"))

In [ ]:
def calculate_technical_indicators(ticker: str, period: str = "6mo") -> dict:
    """計算技術指標：MA20/MA50、RSI(14)、MACD、布林帶，並轉成可讀訊號。

    Args:
        ticker: 股票代碼，例如 'NVDA' 或 '2330.TW'。
        period: 取多久的歷史資料來算，建議至少 '6mo'（MA50 需要 50 個交易日）。

    Returns:
        含各指標數值與訊號字串（ma_trend / rsi_signal / macd_crossover / bb_position）的 dict。
        注意：回傳值裡**沒有** DataFrame，因為這個 dict 要能 JSON 序列化給 LLM 看。
        要畫圖的話用 indicator_dataframe()。
    """
    try:
        df = _indicator_frame(ticker, period)
        if df is None:
            return {"error": f"無法取得 {ticker} 數據"}

        latest = df.iloc[-1]
        return {
            "ticker": ticker,
            "period": period,
            "current_price": _num(latest["Close"]),
            "ma20": _num(latest["MA20"]),
            "ma50": _num(latest["MA50"]),
            "rsi": _num(latest["RSI"]),
            "macd": _num(latest["MACD"], 4),
            "macd_signal": _num(latest["Signal_Line"], 4),
            "macd_histogram": _num(latest["MACD_Histogram"], 4),
            "bb_upper": _num(latest["BB_Upper"]),
            "bb_middle": _num(latest["BB_Middle"]),
            "bb_lower": _num(latest["BB_Lower"]),
            "ma_trend": "📈 多頭" if _num(latest["MA20"]) > _num(latest["MA50"]) else "📉 空頭",
            "rsi_signal": ("過熱" if _num(latest["RSI"]) > 70
                           else "過冷" if _num(latest["RSI"]) < 30 else "中性"),
            "macd_crossover": ("📈 看漲" if _num(latest["MACD"], 4) > _num(latest["Signal_Line"], 4)
                               else "📉 看跌"),
            "bb_position": ("超上軌" if _num(latest["Close"]) > _num(latest["BB_Upper"])
                            else "超下軌" if _num(latest["Close"]) < _num(latest["BB_Lower"])
                            else "正常區間"),
        }
    except Exception as e:
        return {"error": f"計算 {ticker} 指標失敗: {e}"}


def _indicator_frame(ticker: str, period: str = "6mo"):
    """算出帶所有指標欄位的 DataFrame（畫圖用；不進 LLM）。"""
    df = yf.Ticker(ticker).history(period=period)
    if df.empty:
        return None
    df = df.copy()                       # pandas 3 對 chained assignment 更嚴格，先 copy

    df["MA20"] = df["Close"].rolling(20).mean()
    df["MA50"] = df["Close"].rolling(50).mean()

    delta = df["Close"].diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta).clip(lower=0).rolling(14).mean()
    rs = gain / loss.replace(0, float("nan"))   # 避免除以 0 變 inf
    df["RSI"] = 100 - (100 / (1 + rs))

    exp12 = df["Close"].ewm(span=12, adjust=False).mean()
    exp26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = exp12 - exp26
    df["Signal_Line"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_Histogram"] = df["MACD"] - df["Signal_Line"]

    df["BB_Middle"] = df["Close"].rolling(20).mean()
    bb_std = df["Close"].rolling(20).std()
    df["BB_Upper"] = df["BB_Middle"] + bb_std * 2
    df["BB_Lower"] = df["BB_Middle"] - bb_std * 2
    return df


def indicator_dataframe(ticker: str, period: str = "6mo"):
    """公開版：拿帶指標的 DataFrame 來畫圖。"""
    return _indicator_frame(ticker, period)


print("✅ calculate_technical_indicators / indicator_dataframe 已定義")

In [ ]:
# ── 指標數值 + 三張圖 ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

TICKER = "NVDA"          # 想看台積電就改成 "2330.TW"

ind = calculate_technical_indicators(TICKER, "6mo")
print(json.dumps(ind, indent=2, ensure_ascii=False))

df = indicator_dataframe(TICKER, "6mo")
if df is None:
    print("⚠️ 抓不到資料，跳過畫圖")
else:
    fig, ax = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
    # 不加 fontweight="bold"：PingFang TC 沒有 bold 字面，matplotlib 每次畫圖都會往
    # stderr 印一行 `findfont: Failed to find font weight bold, now using 600.`
    fig.suptitle(f"{TICKER} {L('技術指標分析', 'Technical Analysis')}", fontsize=15)

    ax[0].plot(df.index, df["Close"], label=L("收盤價", "Close"), color="black", lw=1.4)
    ax[0].plot(df.index, df["MA20"], label="MA20", color="tab:blue", alpha=0.8)
    ax[0].plot(df.index, df["MA50"], label="MA50", color="tab:orange", alpha=0.8)
    ax[0].fill_between(df.index, df["BB_Upper"], df["BB_Lower"],
                       alpha=0.15, color="gray", label=L("布林帶", "Bollinger"))
    ax[0].set_ylabel(L("價格", "Price"))
    ax[0].set_title(L("價格 / 移動平均 / 布林帶", "Price / MA / Bollinger Bands"))
    ax[0].legend(loc="upper left")
    ax[0].grid(alpha=0.3)

    ax[1].plot(df.index, df["RSI"], color="tab:green", lw=1.4)
    ax[1].axhline(70, color="tab:red", ls="--", alpha=0.7)
    ax[1].axhline(30, color="tab:blue", ls="--", alpha=0.7)
    ax[1].fill_between(df.index, 70, 100, alpha=0.08, color="tab:red")
    ax[1].fill_between(df.index, 0, 30, alpha=0.08, color="tab:blue")
    ax[1].set_ylim(0, 100)
    ax[1].set_ylabel("RSI")
    ax[1].set_title(L("RSI (14)  >70 過熱 / <30 過冷", "RSI (14)  >70 overbought / <30 oversold"))
    ax[1].grid(alpha=0.3)

    ax[2].plot(df.index, df["MACD"], label="MACD", color="tab:blue", lw=1.3)
    ax[2].plot(df.index, df["Signal_Line"], label=L("訊號線", "Signal"),
               color="tab:orange", lw=1.3)
    ax[2].bar(df.index, df["MACD_Histogram"], label=L("柱狀圖", "Histogram"),
              color="gray", alpha=0.5, width=1.0)
    ax[2].axhline(0, color="black", lw=0.8)
    ax[2].set_ylabel("MACD")
    ax[2].set_title("MACD (12, 26, 9)")
    ax[2].legend(loc="upper left")
    ax[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

# 2️⃣ ADK 2.0 核心概念

ADK 2.0 把 1.x 的一堆概念收斂成一組更小的積木：

| 積木 | 是什麼 | 什麼時候用 |
|---|---|---|
| **`BaseNode`** | 2.0 新增的共同基底。`LlmAgent`、`Workflow`、`FunctionNode`、`RemoteA2aAgent` 全都是 `BaseNode` | 這就是為什麼「遠端 A2A agent 可以直接當 Workflow 的節點」 |
| **`LlmAgent`** | 綁一顆 LLM + 一組 tool + instruction | 需要模型推理、需要它自己決定要不要呼叫 tool |
| **`Workflow`** | 用 `edges` 描述的 DAG（§3 主角） | 流程你自己說得清楚時，不要浪費 token 讓模型決定 |
| **`App`** | 把 root node 加上 plugin / cache / resumability 等設定包成一個應用 | 要畫 graph、要 node 狀態快照、要部署 |
| **`Runner`** | 執行器，管 session、artifact、memory | 每次要跑 agent |
| **`Context`** | 一次執行的上下文：`ctx.state`、`ctx.user_content`、`ctx.route` | 在 node 裡讀寫共享狀態 |
| **`Event`** | 執行過程吐出來的事件流 | 想看「內部到底發生什麼事」 |

## `tools` 怎麼寫

ADK 2.0 直接吃**普通 Python 函式**——不用再手動包 `FunctionTool`（想包也可以）。
模型看到的是你的**型別註解 + docstring**，所以 §1 那些函式已經是合格的 tool 了。

## 兩種跑法

- `await runner.run_debug("...")` —— 一行跑起來，適合探索
- `async for ev in runner.run_async(...)` —— 拿到完整事件流，適合教學／除錯／做 UI

In [ ]:
# ── 建一個真的 ADK LlmAgent（不是 mock）────────────────────────────────
from google.adk.agents import LlmAgent
from google.adk.apps import App
from google.adk.runners import InMemoryRunner
from google.genai import types

market_analyst = LlmAgent(
    name="market_analyst",
    model=MODEL,
    description="股市基本面分析師：查報價、歷史走勢與新聞，產出健康度評估。",
    instruction=(
        "你是專業的股市基本面分析師。\n"
        "使用者給你一個股票代碼時，你必須依序呼叫工具取得真實資料：\n"
        "1. get_stock_info 拿現價與基本面\n"
        "2. get_stock_history 拿 6 個月走勢\n"
        "3. get_market_news 拿最近新聞\n"
        "然後用繁體中文寫一份 200 字內的分析，包含：\n"
        "基本面評價、走勢觀察、新聞重點、1-10 分的健康度評分（附理由）。\n"
        "只能引用工具回傳的數字，不要自己編造。"
    ),
    # ⬇️ 直接丟 §1 的 Python 函式進來，ADK 從型別註解 + docstring 生 schema
    tools=[get_stock_info, get_stock_history, get_market_news],
)

print(f"✅ LlmAgent 建立完成: {market_analyst.name}")
print(f"   model      : {market_analyst.model}")
print(f"   tools      : {[getattr(t, '__name__', getattr(t, 'name', t)) for t in market_analyst.tools]}")
print(f"   是 BaseNode : {isinstance(market_analyst, __import__('google.adk.workflow', fromlist=['BaseNode']).BaseNode)}")

In [ ]:
# ── 跑法一：run_debug()，一行就跑 ────────────────────────────────────────
if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    runner = InMemoryRunner(agent=market_analyst, app_name="stockpulse")
    # verbose=True 會把每次 tool call / tool response 都印出來
    events = await runner.run_debug("分析 NVDA", verbose=True)
    print(f"\n（總共 {len(events)} 個 event）")

In [ ]:
# ── 跑法二：run_async()，看完整事件流 ───────────────────────────────────
# 這是理解 ADK 內部運作最重要的一段：一次「分析 NVDA」在底層長成什麼樣子。
if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    runner = InMemoryRunner(agent=market_analyst, app_name="stockpulse")
    session = await runner.session_service.create_session(
        app_name="stockpulse", user_id="student")

    final_text = []
    step = 0
    async for ev in runner.run_async(
        user_id="student",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text="分析 2330.TW")]),
    ):
        step += 1
        for part in (ev.content.parts if ev.content and ev.content.parts else []):
            if part.function_call:
                print(f"[{step:>2}] 🔧 呼叫 tool  {part.function_call.name}"
                      f"({dict(part.function_call.args or {})})")
            elif part.function_response:
                resp = json.dumps(part.function_response.response,
                                  ensure_ascii=False, default=str)
                print(f"[{step:>2}] 📦 tool 回傳  {part.function_response.name} → {resp[:110]}…")
            elif part.text:
                tag = "✅ 最終回覆" if ev.is_final_response() else "💭 模型輸出"
                print(f"[{step:>2}] {tag}")
                if ev.is_final_response():
                    final_text.append(part.text)

    print("\n" + "=" * 68)
    print("\n".join(final_text))

# 3️⃣ ADK 2.0 `Workflow`：把 graph 叫出來

## 為什麼是 `Workflow` 而不是 `SequentialAgent` / `ParallelAgent`

ADK 2.0 起這兩個類別**已經 deprecated**，import 進來用就會看到：

```
DeprecationWarning: SequentialAgent is deprecated in favor of Workflow
                    and will be removed in a future version.
```

原因很直接：`SequentialAgent` 只能串一條線、`ParallelAgent` 只能分岔，
兩者組起來還是很難描述「三路並行完再匯合，然後依結果走不同分支」這種真實流程。
`Workflow` 直接讓你寫 **DAG 的邊**：

```python
Workflow(name="...", edges=[
    (START, a),                  # 進入點
    (a, (b, c, d)),              # fan-out：a 做完，b/c/d 同時開跑
    ((b, c, d), join),           # fan-in：三個都到齊才繼續（join 必須是 JoinNode！）
    (join, decider),
    (decider, {"buy": x, "sell": y, "hold": z}),   # 條件分支
])
```

## 三個一定要知道的坑

**① fan-in 一定要用 `JoinNode`。**
如果你寫 `((b, c, d), merge)` 而 `merge` 是普通的 `@node` 函式，
ADK 會把它當成三條獨立的邊，於是 **`merge` 會被執行三次**（事件路徑是
`merge@1`、`merge@2`、`merge@3`）。用 `JoinNode` 才會「等全部到齊、只跑一次」，
而且它的 output 會自動整理成 `{"上游節點名": 那個節點的輸出}`。

**② `JoinNode` 要是同一個物件實例。**
`edges` 裡兩處都寫 `JoinNode(name="merge")` 會建出兩個物件，驗證直接失敗：
`Duplicate node names found: ['merge']`。先 `merge = JoinNode(name="merge")` 再引用。

**③ 條件分支靠 `ctx.route`，不是 return 值。**
`return "buy"` 沒有用——那只是節點的 output。要寫 `ctx.route = "buy"`。
寫錯的話**不會報錯**，只會印一行 warning 然後流程就默默斷在那個節點，
現場 demo 最容易被這個坑到。

## 節點怎麼傳資料

| 方式 | 寫法 | 說明 |
|---|---|---|
| 共享狀態 | `ctx.state["ticker"] = "NVDA"` | 最通用。下游 node 只要參數名叫 `ticker` 就會自動注入 |
| 參數自動綁定 | `def fetch(ticker: str)` | 預設 `parameter_binding='state'`，按**參數名**去 `ctx.state` 找 |
| LlmAgent 讀狀態 | `instruction="分析 {ticker}，依據 {facts}"` | instruction 裡的 `{key}` 會被 `ctx.state` 的值替換掉 |
| 看上游輸出 | `ev.output`（事件流上） | function node 有；agent node 是 `None`，文字在 `ev.content` |

In [ ]:
# ── show_graph()：一個 helper 同時畫 ADK 和 LangGraph ─────────────────────
# 設計目標：離線可用、失敗會降級不會炸、ADK 跟 LangGraph 同一個呼叫方式。
# 降級順序：本機 graphviz SVG → mermaid PNG（要連網）→ ASCII → 純文字節點列表
import re


def _svg_html(svg: str, max_width: int) -> str:
    """graphviz 產出的 SVG 帶死的 pt 尺寸，大圖會被切掉。改成隨欄寬縮放 + 可橫向捲動。"""
    svg = re.sub(r'(<svg[^>]*?)\swidth="[^"]*"', r"\1", svg, count=1)
    svg = re.sub(r'(<svg[^>]*?)\sheight="[^"]*"', r"\1", svg, count=1)
    svg = svg.replace("<svg", f'<svg style="width:100%;height:auto;max-width:{max_width}px"', 1)
    return f'<div style="overflow-x:auto;max-width:100%">{svg}</div>'


def _adk_svg(obj, agent_state, dark_mode):
    from google.adk.apps import App
    from google.adk.cli.utils.graph_serialization import serialize_app_info
    from google.adk.cli.utils.graph_visualization import plot_workflow_graph

    app = obj if isinstance(obj, App) else App(name=getattr(obj, "name", "app"), root_agent=obj)
    # format="svg" 回傳 str；"png"/"pdf"/"jpg" 回傳 bytes；"dot" 回傳 DOT 原始碼
    return plot_workflow_graph(serialize_app_info(app), agent_state,
                               format="svg", dark_mode=dark_mode)


def _lg_graph(obj):
    return obj.get_graph() if hasattr(obj, "get_graph") else obj


def _lg_svg(obj, dark_mode):
    """用本機 graphviz 把 LangGraph 畫成 SVG（離線可用，而且視覺語言跟 ADK 那張一致）。

    LangGraph 內建的 draw_mermaid_png() 要連 mermaid.ink，draw_png() 要 pygraphviz，
    在教室環境常常兩個都不能用，所以自己從 g.nodes / g.edges 組 Digraph。
    """
    import graphviz

    g = _lg_graph(obj)
    bg, fill, fc, ec = (("#0F172A", "#1E293B", "#F8FAFC", "#94A3B8") if dark_mode
                        else ("#F8FAFC", "#FFFFFF", "#0F172A", "#64748B"))
    dot = graphviz.Digraph()
    dot.attr("graph", bgcolor=bg, pad="0.4", ranksep="0.55", splines="spline")
    dot.attr("node", shape="rect", style="rounded,filled", fillcolor=fill, color=ec,
             fontcolor=fc, fontname="Helvetica", fontsize="12", margin="0.25,0.15")
    dot.attr("edge", color=ec, fontcolor=fc, arrowhead="vee", arrowsize="0.7")

    for name in g.nodes:
        terminal = name in ("__start__", "__end__")
        dot.node(
            name,
            name.strip("_").upper() if terminal else name,
            shape="oval" if terminal else "rect",
            style="filled" if terminal else "rounded,filled",
            fillcolor=("#10B981" if name == "__start__" else "#EF4444") if terminal else fill,
        )
    for e in g.edges:
        dot.edge(e.source, e.target,
                 label=f"  {e.data}" if getattr(e, "data", None) else "",
                 style="dashed" if getattr(e, "conditional", False) else "solid")
    return dot.pipe(format="svg").decode("utf-8")


def show_graph(obj, agent_state=None, dark_mode=False, max_width=900, quiet=True):
    """畫出 ADK App / Workflow / Agent，或 LangGraph 的 CompiledStateGraph。

    Args:
        obj: ADK 的 App / Workflow / BaseAgent，或 LangGraph 編譯後的圖。
        agent_state: 只對 ADK 有效。形狀是 {"nodes": {"節點名": {"status": 0-6}}}，
            用 node_status_from_events() 從事件流蒐集，會把節點按執行狀態上色。
        dark_mode: SVG 自己會畫不透明底色，所以跟你的 Jupyter 主題無關，兩種都看得清楚。
        max_width: 圖片最大寬度（px）。
    Returns:
        實際用到的繪製方式名稱（"svg" / "mermaid_png" / "ascii" / "text"），全失敗回 None。
    """
    from IPython.display import HTML, Image, display

    # ⚠️ 一定要先判斷 ADK：Workflow 也有 .nodes/.edges，用 hasattr 判斷會走錯路
    is_adk = hasattr(obj, "root_agent") or type(obj).__module__.startswith("google.adk")
    is_lg = not is_adk and (hasattr(obj, "get_graph")
                            or (hasattr(obj, "nodes") and hasattr(obj, "edges")))

    for how in (("svg", "text") if is_adk else ("svg", "mermaid_png", "ascii", "text")):
        try:
            if how == "svg":
                svg = _adk_svg(obj, agent_state, dark_mode) if is_adk else _lg_svg(obj, dark_mode)
                display(HTML(_svg_html(svg, max_width)))          # SVG 字串用 HTML/SVG，不能用 Image
            elif how == "mermaid_png":
                display(Image(data=_lg_graph(obj).draw_mermaid_png()))
            elif how == "ascii":
                print(_lg_graph(obj).draw_ascii())
            else:
                if is_lg:
                    g = _lg_graph(obj)
                    print("nodes:", list(g.nodes))
                    for e in g.edges:
                        print(f"  {e.source} → {e.target}")
                else:
                    from google.adk.apps import App
                    from google.adk.cli.utils.graph_serialization import serialize_app_info
                    app = obj if isinstance(obj, App) else App(
                        name=getattr(obj, "name", "app"), root_agent=obj)
                    root = serialize_app_info(app)["root_agent"]
                    gr = root.get("graph")
                    if gr:                                  # Workflow：印節點與邊
                        print("nodes:", [n.get("name") for n in gr.get("nodes", [])])
                        for e in gr.get("edges", []):
                            route = f"  [route={e['route']}]" if e.get("route") else ""
                            print(f"  {e['from_node']['name']} → {e['to_node']['name']}{route}")
                    else:                                   # Agent 樹：遞迴印 sub_agents
                        def walk(n, depth=0):
                            print("  " * depth + f"• {n.get('name')}")
                            for kid in n.get("sub_agents") or []:
                                walk(kid, depth + 1)
                        walk(root)
            if how != "svg":
                print(f"（用了備援方式：{how}）")
            return how
        except Exception as ex:
            if not quiet:
                print(f"[show_graph] {how} 不可用：{type(ex).__name__}: {str(ex)[:100]}")
    return None


def node_status_from_events(events) -> dict:
    """從事件流蒐集節點狀態快照，餵給 show_graph(..., agent_state=...) 就會上色。

    前提：App 要開 resumability，Workflow 才會送出狀態快照事件：
        App(name=..., root_agent=wf, resumability_config=ResumabilityConfig(is_resumable=True))
    沒開的話 ev.actions.agent_state 永遠是 None，圖上每個節點都會是白的。

    實測每一份快照都是「當下的完整節點表」，所以最後一份通常就等於累積結果；
    用 update() 一路累積只是保險寫法（跟事件順序無關，也不會漏節點）。
    """
    snapshot = {}
    for ev in events:
        state = getattr(getattr(ev, "actions", None), "agent_state", None)
        if state and "nodes" in state:
            snapshot.update(state["nodes"])
    return {"nodes": snapshot}


# NodeStatus 是 int Enum；傳字串（"completed"）會被靜默當成 INACTIVE（白色），是個坑
from google.adk.workflow._node_status import NodeStatus

STATUS_NAME = {s.value: s.name for s in NodeStatus}
print("✅ show_graph() / node_status_from_events() 已定義")
print("   NodeStatus:", STATUS_NAME)
print("   顏色：COMPLETED 綠 / RUNNING 橘 / FAILED 紅 / WAITING 紫 / INACTIVE 白")

In [ ]:
# ── Market Analyst 的節點：4 個 function node + 1 個 LlmAgent + 1 個分支 ──
from google.adk.agents import Context, LlmAgent
from google.adk.apps import App, ResumabilityConfig
from google.adk.runners import InMemoryRunner
from google.adk.workflow import JoinNode, START, Workflow, node
from google.genai import types

TW_RE = __import__("re").compile(r"\b(\d{4,6}\.TW[O]?)\b")
US_RE = __import__("re").compile(r"\b([A-Z]{1,5})\b")


@node
def parse_ticker(ctx: Context) -> dict:
    """從使用者訊息裡找出股票代碼，寫進 ctx.state 給下游用。"""
    text = ""
    if ctx.user_content and ctx.user_content.parts:
        text = " ".join(p.text or "" for p in ctx.user_content.parts)
    upper = text.upper()
    tw = TW_RE.search(upper)
    us = US_RE.search(upper)
    ticker = tw.group(1) if tw else (us.group(1) if us else "NVDA")

    ctx.state["ticker"] = ticker           # ← 下游 node 的 `ticker` 參數會自動綁到這裡
    print(f"   🔎 parse_ticker    → {ticker}")
    return {"ticker": ticker}


# ⬇️ 這三個是 fan-out 的分支，會「同時」執行
# 這三個節點的簽名同時要 `ticker: str`（從 ctx.state 按參數名自動綁定）
# 和 `ctx: Context`（用來寫回 state）。混用是合法的。
@node
def fetch_quote(ticker: str, ctx: Context) -> dict:
    """抓即時報價與基本面。"""
    data = get_stock_info(ticker)
    ctx.state["quote"] = data                    # 寫回 state 給下游 build_facts 用
    print(f"   💵 fetch_quote     → {data.get('current_price')} {data.get('currency', '')}")
    return data


@node
def fetch_indicators(ticker: str, ctx: Context) -> dict:
    """算技術指標。"""
    data = calculate_technical_indicators(ticker, "6mo")
    ctx.state["indicators"] = data
    print(f"   📐 fetch_indicators→ RSI {data.get('rsi')} / {data.get('ma_trend')}")
    return data


@node
def fetch_news(ticker: str, ctx: Context) -> dict:
    """抓最近新聞。"""
    data = get_market_news(ticker, top_n=3)
    ctx.state["news"] = data
    print(f"   📰 fetch_news      → {len(data.get('items', []))} 則")
    return data


# ⬇️ fan-in 必須是 JoinNode，而且只能建一個實例（下面兩條邊都引用它）
gather = JoinNode(name="gather")


@node
def build_facts(ctx: Context) -> dict:
    """把三路結果整理成一段文字，寫進 state 供 LlmAgent 的 instruction 取用。

    注意這裡**不重抓資料**——三個上游節點已經把結果寫進 ctx.state 了，直接讀就好。
    這就是 fan-out 節省時間的意義：三個 I/O 同時做，這裡只負責組裝。
    """
    ticker = ctx.state.get("ticker", "?")
    quote = ctx.state.get("quote") or {}
    ind = ctx.state.get("indicators") or {}
    news = ctx.state.get("news") or {}

    facts = (
        f"報價：現價 {quote.get('current_price')} {quote.get('currency', '')}，"
        f"前收 {quote.get('previous_close')}，P/E {quote.get('pe_ratio')}，"
        f"股利率 {quote.get('dividend_yield_pct')}%，"
        f"52週區間 {quote.get('week52_low')}~{quote.get('week52_high')}\n"
        f"技術面：MA20 {ind.get('ma20')} / MA50 {ind.get('ma50')}（{ind.get('ma_trend')}），"
        f"RSI {ind.get('rsi')}（{ind.get('rsi_signal')}），"
        f"MACD {ind.get('macd_crossover')}，布林帶 {ind.get('bb_position')}\n"
        f"新聞：" + " | ".join(i["title"][:60] for i in news.get("items", [])) or "新聞：無"
    )
    ctx.state["facts"] = facts
    ctx.state["rsi"] = ind.get("rsi", 50)

    # 條件分支：靠 ctx.route，不是 return 值！
    rsi = ind.get("rsi") or 50
    ctx.route = "extreme" if (rsi > 70 or rsi < 30) else "normal"
    print(f"   🧩 build_facts     → route={ctx.route}（RSI {rsi}）")
    return {"facts_len": len(facts), "route": ctx.route}


analyst = LlmAgent(
    name="analyst",
    model=MODEL,
    description="把彙整好的資料寫成基本面分析報告。",
    # instruction 裡的 {ticker} / {facts} 會被 ctx.state 對應的值替換
    instruction=(
        "你是股市基本面分析師。以下是 {ticker} 的真實資料：\n\n{facts}\n\n"
        "請用繁體中文寫 180 字內的分析，包含：基本面評價、技術面觀察、"
        "新聞重點、健康度評分（1-10，附一句理由）。只引用上面的數字，不要編造。"
    ),
)

risk_analyst = LlmAgent(
    name="risk_analyst",
    model=MODEL,
    description="RSI 進入極端區時，額外做一份超買/超賣風險提醒。",
    instruction=(
        "{ticker} 的 RSI 是 {rsi}，已進入極端區。資料：\n\n{facts}\n\n"
        "請用繁體中文寫 180 字內的分析，並**特別強調**超買或超賣的風險、"
        "可能的均值回歸，以及短線該注意什麼。最後給健康度評分（1-10）。"
    ),
)

print("✅ 6 個節點已定義：parse_ticker / fetch_quote / fetch_indicators / fetch_news")
print("                  / gather(JoinNode) / build_facts → analyst | risk_analyst")

In [ ]:
# ── 組成 Workflow ────────────────────────────────────────────────────────
market_analyst_wf = Workflow(
    name="market_analyst_wf",
    description="抓報價／指標／新聞三路並行，匯合後依 RSI 走一般或極端分析。",
    edges=[
        (START, parse_ticker),                                    # 進入點
        (parse_ticker, (fetch_quote, fetch_indicators, fetch_news)),   # fan-out：三路同時
        ((fetch_quote, fetch_indicators, fetch_news), gather),    # fan-in：JoinNode 等齊
        (gather, build_facts),
        (build_facts, {"normal": analyst, "extreme": risk_analyst}),   # 條件分支
    ],
    max_concurrency=3,        # 同時最多跑 3 個節點（None = 不限）
)

# is_resumable=True 才會送出節點狀態快照事件 → 才能畫出「上色版」的 graph
market_analyst_app = App(
    name="stockpulse_market",
    root_agent=market_analyst_wf,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

print(f"✅ Workflow: {market_analyst_wf.name}")
print(f"   節點數 {len(market_analyst_wf.graph.nodes)} / 邊數 {len(market_analyst_wf.graph.edges)}")
for e in market_analyst_wf.graph.edges:
    route = f"   [route={e.route}]" if e.route else ""
    print(f"   {e.from_node.name:>18} → {e.to_node.name}{route}")

### 👀 把 graph 叫出來

`plot_workflow_graph()` 是 ADK 自己 dev server 用的同一支繪圖程式，
直接餵 `serialize_app_info(app)` 就會畫出來。圖上讀得到的資訊：

- **橢圓** = `START` / `END`
- **矩形** = function node
- **菱形** = 有條件分支的節點（邊上會標 `route=`）
- **虛線 🔧** = LlmAgent 的 tool（自動補上，不用你畫）
- **⊷ 紫色** = 巢狀 workflow / 橋接進來的其他框架

In [ ]:
# 靜態結構圖（還沒跑，所以節點都是白的）
show_graph(market_analyst_app)

In [ ]:
# ── 跑 Workflow，順便把節點狀態收下來 ───────────────────────────────────
QUERY = "幫我分析 NVDA"

analyst_events = []
market_report = ""

if not HAS_KEY:
    print("SKIPPED：沒有 API Key（graph 已經畫出來了，那部分不需要 key）")
else:
    runner = InMemoryRunner(app=market_analyst_app)
    session = await runner.session_service.create_session(
        app_name="stockpulse_market", user_id="student")

    print(f"🚀 執行 {market_analyst_wf.name}：{QUERY}\n")
    async for ev in runner.run_async(
        user_id="student",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=QUERY)]),
    ):
        analyst_events.append(ev)

        path = getattr(getattr(ev, "node_info", None), "path", None)
        # function node → 結果在 ev.output；agent node → ev.output 是 None，文字在 ev.content
        if ev.output is not None:
            print(f"   ✔ {path:<42} output={json.dumps(ev.output, ensure_ascii=False, default=str)[:90]}")
        elif ev.content and ev.content.parts:
            text = " ".join(p.text or "" for p in ev.content.parts).strip()
            if text:
                print(f"   ✔ {path:<42} text={len(text)} 字")
                market_report = text

    print(f"\n✅ 完成，共 {len(analyst_events)} 個 event")

### 🎨 同一張 graph，用執行狀態上色

這是 ADK 2.0 最實用的除錯工具：跑完之後把 `agent_state` 餵回去，
**綠色 = 執行完、白色 = 沒被執行到**。條件分支哪一邊真的走了，一眼就看出來。

In [ ]:
if not analyst_events:
    print("SKIPPED：上面沒有跑，沒有狀態可以上色")
else:
    state = node_status_from_events(analyst_events)
    print("節點狀態：")
    for name, info in state["nodes"].items():
        print(f"   {name:<20} {STATUS_NAME.get(info.get('status'), info.get('status'))}")
    print("\n（沒出現在上面的節點 = 沒被執行到 = 圖上是白色）\n")
    show_graph(market_analyst_app, agent_state=state)

In [ ]:
# ── 最終報告 ─────────────────────────────────────────────────────────────
from IPython.display import Markdown, display

if market_report:
    display(Markdown(f"""### 🏦 Market Analyst 報告（ADK 2.0 Workflow）

{market_report}

---
*由 `market_analyst_wf` 產生：3 路並行抓資料 → JoinNode 匯合 → 依 RSI 分支 → LlmAgent 撰寫*
"""))
else:
    print("（沒有報告：可能沒 API Key，或上面的 cell 還沒執行）")

# 4️⃣ LangGraph Trading Strategist + `LangGraphAgent` 橋接

## 為什麼還要 LangGraph？

ADK `Workflow` 和 LangGraph 解的是同一類問題，但生態不同。真實專案常常是
「某個團隊已經有一堆 LangGraph 資產」，這時候重寫不划算——ADK 2.0 提供
`LangGraphAgent`，**把編譯好的 LangGraph 直接包成一個 ADK node**，
於是它可以塞進 `Workflow`、當 `LlmAgent` 的 sub_agent、也能用 `to_a2a()` 發佈成 A2A 服務。

## 這一節的三個坑

**① `msg.content` 在 Gemini 3.x 不是字串。**
`langchain-google-genai` 對 `gemini-2.5-*` 回傳 `str`，
但對 `gemini-3.x` 回傳 **list of content blocks**
（`[{'type':'text','text':…,'extras':{'signature':…}}]`）。
直接 f-string 進去會印出一坨 dict。用 `msg.text`（langchain-core 的 str property），兩種都對。

**② 橋接要求「圖的最後一則訊息 content 必須是純字串」。**
`LangGraphAgent` 內部做 `types.Part.from_text(text=final_state["messages"][-1].content)`，
拿到 list 會直接 pydantic ValidationError。所以**在節點裡就要正規化**，不是事後補。

**③ `add_messages` reducer 只回傳「新訊息」。**
寫 `return {"messages": state["messages"] + [new]}` 會讓歷史每一步都翻倍複製。
正確是 `return {"messages": [new]}`。

**④ 送給 Gemini 3.x 的對話不能以 assistant 結尾。**
這是最難查的一個。鏈式節點天生就會這樣：node 2 把 AIMessage 存進 state，
node 3 再 `llm.invoke([system] + state["messages"])` → 對話最後一則是 assistant。
Gemini 3.x 遇到這種對話會**回 0 個 output token**（`finish_reason=STOP`、`content=[]`），
它認為「助理已經講完了」，就算 SystemMessage 明確要求它做風險評估也一樣。
舊版教材用 gemini-2.0-flash 剛好沒事，換 3.x 之後 `risk_assessment` 整段變空白。
解法是下面的 `to_user_turn()`：把上游的 AI 產出包成 `HumanMessage` 再送進去。

In [ ]:
# ── LangGraph 的三個節點 ────────────────────────────────────────────────
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START as LG_START, MessagesState, StateGraph


def text_of(msg) -> str:
    """從 langchain message 取純文字。

    Gemini 2.5 給 str、Gemini 3.x 給 list of blocks，msg.text 兩種都處理得好。
    """
    text = getattr(msg, "text", None)
    if isinstance(text, str) and text:
        return text
    content = getattr(msg, "content", msg)
    if isinstance(content, str):
        return content
    if isinstance(content, list):          # [{'type':'text','text':...}, ...]
        return "".join(b.get("text", "") for b in content if isinstance(b, dict))
    return str(content)


def to_user_turn(messages: list) -> list:
    """把對話中的 AI 回覆改寫成 user 回合，讓 LLM 呼叫「以 user 結尾」。

    ⚠️ 這個 helper 解掉一個很難查的坑：**Gemini 3.x 收到「最後一則是 assistant」
    的對話時，會回 0 個 output token**（`finish_reason=STOP`、`content=[]`），
    因為它認為助理已經講完了——就算你的 SystemMessage 明確要求它做下一件事也一樣。

    LangGraph 的鏈式節點天生就會踩到：node 2 產出 AIMessage 進 state，
    node 3 再 `llm.invoke([system] + state["messages"])`，對話就以 assistant 結尾。
    舊版教材用 gemini-2.0-flash 剛好沒事，換到 3.x 就整段變空白。

    解法是把上游的 AI 產出包成 HumanMessage 再送進去。
    （注意：只影響「送給 LLM 的那份 list」，state 裡還是照常存 AIMessage。）
    """
    turns = []
    for m in messages:
        if isinstance(m, AIMessage):
            tag = getattr(m, "name", None) or "前一步"
            turns.append(HumanMessage(content=f"【{tag} 的產出】\n{text_of(m)}"))
        else:
            turns.append(m)
    return turns


llm = None
if HAS_KEY:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.7,
                                 google_api_key=GOOGLE_API_KEY)
    print(f"✅ ChatGoogleGenerativeAI 已初始化（{MODEL}）")
else:
    print("⚠️ 沒 key，LangGraph 的 LLM 節點會回固定字串（圖還是畫得出來、跑得動）")


def compute_indicators_node(state: MessagesState) -> dict:
    """節點 1：從訊息裡找股票代碼，算技術指標，塞回對話。"""
    user_msg = text_of(state["messages"][-1])
    upper = user_msg.upper()
    tw, us = TW_RE.search(upper), US_RE.search(upper)
    ticker = tw.group(1) if tw else (us.group(1) if us else "NVDA")

    ind = calculate_technical_indicators(ticker, "6mo")
    print(f"   📐 compute_indicators → {ticker} RSI={ind.get('rsi')} {ind.get('ma_trend')}")

    # ✅ 只回傳「新產生的訊息」，add_messages 會自動 append。
    #    寫成 state["messages"] + [msg] 會讓對話歷史指數級複製。
    return {"messages": [HumanMessage(
        content=f"【{ticker} 技術指標】\n{json.dumps(ind, indent=2, ensure_ascii=False)}"
    )]}


def strategy_analysis_node(state: MessagesState) -> dict:
    """節點 2：產出交易策略與訊號。"""
    if llm is None:
        return {"messages": [AIMessage(content="（沒有 API Key，略過策略生成）")]}

    system = SystemMessage(content=(
        "你是專業量化交易策略師。依據提供的技術指標，用繁體中文輸出：\n"
        "1. 技術分析摘要（80 字內）\n2. 交易訊號（BUY / SELL / HOLD）\n"
        "3. 建議進場價位\n4. 止損價位（-5%~-8%）\n5. 止盈價位（+10%~+15%）\n"
        "6. 信心度（0-100%）\n7. 注意事項\n全部合起來不要超過 300 字。"
    ))
    resp = llm.invoke([system] + to_user_turn(state["messages"]))
    print(f"   🧠 strategy_analysis  → {len(text_of(resp))} 字")
    # ⚠️ 一定要正規化成純字串：ADK 的 LangGraphAgent 橋接會直接把 content 丟給
    #    types.Part.from_text()，拿到 list of blocks 會 ValidationError
    return {"messages": [AIMessage(content=text_of(resp) or "（模型未回覆文字）",
                                   name="strategy")]}


def risk_assessment_node(state: MessagesState) -> dict:
    """節點 3：風險審查（保守原則），這是圖的最後一站。"""
    if llm is None:
        return {"messages": [AIMessage(content="（沒有 API Key，略過風險評估）")]}

    system = SystemMessage(content=(
        "你是風險管理專家。審查前面的交易策略，用繁體中文輸出：\n"
        "1. 風險等級（LOW / MEDIUM / HIGH）\n2. 最大可能損失\n3. 風險因子清單\n"
        "4. 建議部位大小（佔總資產 %）\n5. 最終建議（是否執行）\n"
        "基於保守原則，全部不要超過 300 字。"
    ))
    # ← to_user_turn 是必要的：上一個節點留下 AIMessage，不轉換的話 Gemini 3.x 會回空白
    resp = llm.invoke([system] + to_user_turn(state["messages"]))
    print(f"   🛡️ risk_assessment    → {len(text_of(resp))} 字")
    return {"messages": [AIMessage(content=text_of(resp) or "（模型未回覆文字）",
                                   name="risk")]}


print("✅ text_of / to_user_turn / 三個 LangGraph 節點已定義")

In [ ]:
# ── 建圖並編譯 ───────────────────────────────────────────────────────────
builder = StateGraph(MessagesState)
builder.add_node("compute_indicators", compute_indicators_node)
builder.add_node("strategy_analysis", strategy_analysis_node)
builder.add_node("risk_assessment", risk_assessment_node)

builder.add_edge(LG_START, "compute_indicators")
builder.add_edge("compute_indicators", "strategy_analysis")
builder.add_edge("strategy_analysis", "risk_assessment")
builder.add_edge("risk_assessment", END)

trading_graph = builder.compile()
print(f"✅ LangGraph 編譯完成: {type(trading_graph).__name__}")
print(f"   節點: {list(trading_graph.get_graph().nodes)}")

### 👀 把 LangGraph 的 graph 也叫出來

同一個 `show_graph()` 就行。它會用本機 graphviz 畫（離線可用），
所以跟 §3 那張 ADK 圖的視覺語言一致，方便並排對照。

LangGraph 自己也有三種畫法，但在教室環境常常不能用：

| 方法 | 回傳 | 限制 |
|---|---|---|
| `draw_mermaid()` | mermaid 文字 | ✅ 離線可用 |
| `draw_ascii()` | ASCII 圖 | 需要 `grandalf` |
| `draw_mermaid_png()` | PNG bytes | ⚠️ 要連 mermaid.ink |
| `draw_png()` | PNG bytes | ❌ 要 `pygraphviz`（要編譯，常裝不起來） |

In [ ]:
show_graph(trading_graph)

# mermaid 原始碼（可以貼到 GitHub README、Notion、mermaid.live）
print("\n─── mermaid 原始碼 ───")
print(trading_graph.get_graph().draw_mermaid())

In [ ]:
# ── 直接跑 LangGraph，用 stream 看每個節點的產出 ────────────────────────
# stream_mode="updates" 每個節點跑完就吐一次，很適合展示流程。
# 注意：LangGraph 1.2 的同步 invoke/stream 在 notebook 的 event loop 裡也能用，
# 所以這個 cell 不需要 await。
lg_result = None

if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    print("🚀 執行 LangGraph trading_graph：分析 2330.TW\n")
    lg_state = {"messages": [HumanMessage(content="請分析 2330.TW 的交易策略")]}

    # stream_mode=["updates","values"] 一次跑完就同時拿到「逐節點進度」和「最終狀態」，
    # 不用先 stream 再 invoke 跑第二次（那會多花一倍的 token）。
    for mode, chunk in trading_graph.stream(lg_state, stream_mode=["updates", "values"]):
        if mode == "updates":
            for node_name, update in chunk.items():
                print(f"   ▸ 節點 {node_name} 產出 {len(update.get('messages', []))} 則訊息")
        else:
            lg_result = chunk          # values 模式每步給完整 state，最後一次就是結果

    print(f"\n✅ 完成，對話共 {len(lg_result['messages'])} 則訊息")
    print(f"   最後一則型別: {type(lg_result['messages'][-1]).__name__}")
    print(f"   content 型別: {type(lg_result['messages'][-1].content).__name__} "
          f"← Gemini 3.x 會是 list，所以要用 text_of()")

In [ ]:
# ── 完整輸出 ─────────────────────────────────────────────────────────────
if lg_result is None:
    print("SKIPPED")
else:
    parts = []
    for i, msg in enumerate(lg_result["messages"]):
        tag = getattr(msg, "name", None) or type(msg).__name__
        parts.append(f"#### 訊息 {i + 1} — `{tag}`\n\n{text_of(msg)}\n")
    display(Markdown("### 📈 LangGraph Trading Strategist 完整輸出\n\n"
                     + "\n---\n".join(parts)))

### 🔗 橋接：LangGraph → ADK

`LangGraphAgent(graph=…)` 一行就把 LangGraph 變成 ADK 的一等公民。

橋接的實際行為（讀 `google/adk/agents/langgraph_agent.py` 得到）：

1. 把 ADK session 的事件轉成 langchain 的 `HumanMessage` / `AIMessage`
2. 圖的 state 是空的時候，把 `instruction` 當 `SystemMessage` 塞在最前面
3. `await graph.ainvoke({"messages": messages})`
4. 取 `final_state["messages"][-1].content` 當回覆

所以：**圖的 state 一定要有 `messages` 這個 key**（用 `MessagesState` 就對了），
而且最後一則訊息的 `content` 必須是純字串。

一個要知道的限制：橋接**只吐一個 ADK event**（整張圖算一步），
看不到 LangGraph 內部的逐節點進度。想展示逐節點就用上面的 `graph.stream()`。

In [ ]:
# ── LangGraph → ADK agent ────────────────────────────────────────────────
from google.adk.agents.langgraph_agent import LangGraphAgent
from google.adk.workflow import BaseNode

trading_strategist = LangGraphAgent(
    name="trading_strategist",
    description="技術分析 + 交易訊號 + 風險評估（內部是 LangGraph 三節點流程）。",
    instruction="你是 StockPulse 的交易策略團隊，收到股票代碼就做完整的技術分析與風險評估。",
    graph=trading_graph,
)

print(f"✅ LangGraphAgent: {trading_strategist.name}")
print(f"   是 ADK BaseNode  : {isinstance(trading_strategist, BaseNode)}")
print(f"   → 所以它可以：當 Workflow 的節點、當 LlmAgent 的 sub_agent、用 to_a2a() 發佈")
print(f"   MRO: {' → '.join(c.__name__ for c in type(trading_strategist).__mro__[:4])}")

In [ ]:
# ── 用 ADK Runner 跑它，證明橋接真的成立 ────────────────────────────────
bridge_output = ""

if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    runner = InMemoryRunner(agent=trading_strategist, app_name="bridge_demo")
    session = await runner.session_service.create_session(
        app_name="bridge_demo", user_id="student")

    print("🚀 透過 ADK Runner 執行 LangGraph（注意：只會有一個 event）\n")
    n = 0
    async for ev in runner.run_async(
        user_id="student",
        session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text="分析 NVDA")]),
    ):
        n += 1
        text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else []))
        print(f"   ADK Event #{n}  author={ev.author}  final={ev.is_final_response()}  "
              f"{len(text)} 字")
        if text:
            bridge_output = text

    print(f"\n✅ LangGraph 的輸出已經變成 ADK Event（共 {n} 個）")
    print("─" * 68)
    print(bridge_output[:600] + ("…" if len(bridge_output) > 600 else ""))

In [ ]:
# ── 順手看一眼：橋接後在 ADK graph 裡長什麼樣 ──────────────────────────
# LangGraphAgent 在 ADK 的圖上是一個 ⊷ 紫色方框（內部的 LangGraph 不會被展開，
# 這是刻意的：兩層分開畫，教學上反而更清楚）。
demo_wf = Workflow(
    name="bridge_demo_wf",
    description="示範 LangGraphAgent 當 ADK Workflow 的節點。",
    edges=[(START, parse_ticker), (parse_ticker, trading_strategist)],
)
show_graph(demo_wf)

# 5️⃣ 真正的 A2A Protocol

## 舊版教材的 A2A 是假的

舊版是這樣寫的：

```python
market_analyst_card = {"name": "...", "endpoints": {...}}   # 一個 dict
async def stockpulse_coordinator(ticker):                   # 一個 async 函式
    print("📡 [A2A Task] 發送任務給 Market Analyst")          # 一行 print
    market_result = await market_analyst_agent.analyze(ticker)   # 直接呼叫本地物件
```

沒有 HTTP、沒有 AgentCard 端點、沒有 Task 生命週期、沒有協定。
換句話說：**把 agent 部署到另一台機器上，這段程式碼一行都不能用**。

## 這一節做真的

| 步驟 | 做什麼 | 用到的東西 |
|---|---|---|
| 1 | 把 ADK agent 變成 A2A server | `to_a2a(agent)` → Starlette app |
| 2 | 背景 thread 起 uvicorn | 真的監聽 TCP port |
| 3 | 抓真的 AgentCard | `GET /.well-known/agent-card.json` |
| 4 | 裸 JSON-RPC 呼叫 | `POST /` `{"method":"message/send"}`，看得到 Task 生命週期 |
| 5 | SSE streaming | `message/stream`，看 `submitted → working → artifact → completed` |
| 6 | ADK 原生 client | `RemoteA2aAgent(agent_card=<url>)` |
| 7 | **Coordinator Workflow** | 兩個 `RemoteA2aAgent` 當節點，fan-out → JoinNode → 綜合 |

## A2A 核心概念

- **AgentCard** — agent 的自我介紹（名字、能力 skills、支援的傳輸方式、endpoint URL）。放在固定路徑 `/.well-known/agent-card.json`，client 靠它做服務發現。
- **Task** — 一次委派的工作，有生命週期：`submitted → working → completed / failed / canceled`。
- **Message / Part** — 訊息由 parts 組成（text / data / file）。
- **Transport** — 這裡是 JSON-RPC 2.0 over HTTP；`message/send` 同步、`message/stream` 走 SSE。

## ⚠️ 這一節的環境重點：不要用 `nest_asyncio`

舊版教材第一個 cell 就 `nest_asyncio.apply()`。在這一版**必須拿掉**，
因為在 Python 3.14 + uvicorn 0.52 + ipykernel 7 的組合下它會弄壞兩件事：

1. uvicorn 啟動時呼叫 `asyncio_run(..., loop_factory=...)` →
   `TypeError: run() got an unexpected keyword argument 'loop_factory'` → **背景 server 根本起不來**
2. `RemoteA2aAgent` 解析 AgentCard 時 anyio/sniffio 認不出被 patch 過的 loop →
   `unknown async library, or not in async context`

ipykernel 7 本來就支援 cell 裡直接 `await`，不需要 `nest_asyncio`。

In [ ]:
# ── A2A server 的基礎設施 ────────────────────────────────────────────────
import socket
import threading
import time

import httpx
import uvicorn

from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents.remote_a2a_agent import AGENT_CARD_WELL_KNOWN_PATH, RemoteA2aAgent

print(f"AgentCard 的固定路徑: {AGENT_CARD_WELL_KNOWN_PATH}")

# 模組級 singleton：重跑這個 cell 不會噴 "address already in use"
A2A_SERVERS = globals().setdefault("A2A_SERVERS", {})


def free_port() -> int:
    """跟 OS 要一個沒被佔用的 port（bind 0 讓 OS 挑）。"""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def serve_a2a(agent, key: str, agent_card=None, host: str = "127.0.0.1",
              timeout: float = 30.0) -> dict:
    """把 ADK agent（或 Workflow）發佈成 A2A server，跑在背景 thread。

    Returns:
        {"srv", "thread", "port", "base", "rpc", "card_url", "card"}
    """
    if key in A2A_SERVERS:                     # 已經在跑了，直接回傳
        print(f"   ♻️  {key} 已在 {A2A_SERVERS[key]['base']} 執行中")
        return A2A_SERVERS[key]

    port = free_port()
    # rpc_path 預設 ""，所以 JSON-RPC 端點就是 app 根目錄 http://host:port/
    app = to_a2a(agent, host=host, port=port, agent_card=agent_card)
    srv = uvicorn.Server(uvicorn.Config(app, host=host, port=port, log_level="error"))
    srv.install_signal_handlers = lambda: None      # 非主執行緒不能裝 signal handler
    thread = threading.Thread(target=srv.run, daemon=True, name=f"a2a-{key}")
    thread.start()

    base = f"http://{host}:{port}"
    card_url = f"{base}{AGENT_CARD_WELL_KNOWN_PATH}"

    # ⚠️ to_a2a() 的路由是在 Starlette lifespan 裡掛上的，所以「port 開了」不等於
    #    「card 拿得到」。一定要 poll AgentCard，不能 time.sleep 猜。
    deadline = time.time() + timeout
    card = None
    while time.time() < deadline:
        if not thread.is_alive():
            raise RuntimeError(f"{key} 的 uvicorn thread 在啟動時就死了")
        try:
            r = httpx.get(card_url, timeout=2.0)
            if r.status_code == 200:
                card = r.json()
                break
        except Exception:
            pass
        time.sleep(0.2)
    if card is None:
        raise TimeoutError(f"{key} 在 {timeout}s 內沒有提供 AgentCard")

    info = {"srv": srv, "thread": thread, "port": port, "base": base,
            "rpc": base + "/", "card_url": card_url, "card": card}
    A2A_SERVERS[key] = info
    print(f"   ✅ {key:<20} {base}  (skills: {len(card.get('skills', []))})")
    return info


def stop_a2a(key: str = None):
    """關掉背景 server（key=None 就全關）。port 會立刻釋放，可以重新啟動。"""
    for k in ([key] if key else list(A2A_SERVERS)):
        info = A2A_SERVERS.pop(k, None)
        if info:
            info["srv"].should_exit = True
            info["thread"].join(timeout=5)
            print(f"   🛑 {k} 已關閉（thread alive={info['thread'].is_alive()}）")


print("✅ serve_a2a() / stop_a2a() / free_port() 已定義")

In [ ]:
# ── 把 §3 和 §4 的兩個 agent 都發佈成 A2A server ────────────────────────
# 注意：A2A 服務的最後一個節點必須是「會產出文字的 agent」。
#   如果 Workflow 最後一個節點是普通的 @node 函式，它回傳的 dict 不會變成 A2A 的
#   artifact，Task 會一直停在 working 狀態拿不到答案。
#   market_analyst_wf 結尾是 LlmAgent ✅，trading_strategist 是 LangGraphAgent ✅
print("🚀 啟動 A2A servers…")
ma = serve_a2a(market_analyst_wf, "market-analyst")
ts = serve_a2a(trading_strategist, "trading-strategist")

print(f"\nMarket Analyst      RPC: {ma['rpc']}")
print(f"                    Card: {ma['card_url']}")
print(f"Trading Strategist  RPC: {ts['rpc']}")
print(f"                    Card: {ts['card_url']}")

### 📇 真的 AgentCard

這不是我們手寫的 dict，是 **HTTP GET 回來的**。ADK 會從 agent 的
`name` / `description` / `tools` / `graph` 自動生出 `skills`。

> **a2a-sdk 1.x 的欄位變動**（教材裡最容易踩的地方）
> 舊的 0.3.x 是 `card["url"]` 和 `card["protocolVersion"]`。
> 1.x 改成 protobuf 結構，endpoint 搬到 **`card["supportedInterfaces"][0]["url"]`**。
> 照舊寫法讀 `card["url"]` 會直接 `KeyError`。

In [ ]:
for key, info in A2A_SERVERS.items():
    resp = httpx.get(info["card_url"], timeout=10)
    card = resp.json()
    print("=" * 74)
    print(f"GET {info['card_url']}   →  HTTP {resp.status_code}")
    print("=" * 74)
    print(json.dumps(card, indent=2, ensure_ascii=False))
    print()

print("─" * 74)
print("欄位解讀：")
c = A2A_SERVERS["market-analyst"]["card"]
print(f"  name                     {c['name']}")
print(f"  version                  {c['version']}")
print(f"  supportedInterfaces[0]   {c['supportedInterfaces'][0]}")
print(f"      ↑ 1.x 的 endpoint 在這裡，不是頂層的 card['url']")
print(f"  capabilities             {c['capabilities']}")
print(f"      ↑ streaming=false 是 ADK 自動產生 card 的預設值，下面會示範怎麼打開")
print(f"  defaultInputModes        {c['defaultInputModes']}")
print(f"  skills                   {len(c['skills'])} 個：")
for s in c["skills"]:
    print(f"      • {s['id']:<38} tags={s['tags']}")

### 📡 裸 JSON-RPC：真的把協定看清楚

ADK 起的 A2A server 在**同一個端點**上同時講兩種方言：

| 方言 | method 名稱 | params 格式 | 需要的 header |
|---|---|---|---|
| **0.3 相容**（推薦教學用） | `message/send`、`message/stream` | 人類可讀，state 是 `submitted`/`working`/`completed` | 不用 |
| 1.x 原生 | `SendMessage`、`SendStreamingMessage`、`GetTask`… | protobuf-JSON，state 是 `TASK_STATE_*` | **`A2A-Version: 1.0`** |

漏掉 `A2A-Version: 1.0` 而用 1.x 方言的 method，會拿到
`-32009 VERSION_NOT_SUPPORTED`。下面兩種都示範一次。

答案在哪裡：0.3 方言是 `result.artifacts[0].parts[0].text`。
另外 `result.history` 是遠端 agent 這次 Task 的完整訊息流。
**如果遠端 agent 有呼叫 tool**，軌跡也會在裡面，以
`kind:"data"` + `metadata.adk_type = function_call / function_response` 的形式出現——
這是 A2A 很好用的一點：遠端 agent 內部做了什麼，你在 client 端看得到。
（我們這裡的 `market_analyst_wf` 是用 function node 抓資料、LlmAgent 只負責寫報告，
所以它的 history 只有 user 與 agent 兩則文字；§2 那個會呼叫 tool 的 `market_analyst`
發佈成 A2A 的話，history 就會多出 function_call / function_response 的軌跡。）

In [ ]:
import uuid

if not HAS_KEY:
    print("SKIPPED：沒有 API Key（遠端 agent 需要呼叫 Gemini）")
else:
    RPC = A2A_SERVERS["market-analyst"]["rpc"]

    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "message/send",
        "params": {
            "message": {
                "role": "user",
                "parts": [{"kind": "text", "text": "分析 NVDA"}],
                "messageId": uuid.uuid4().hex,
                "kind": "message",
            }
        },
    }

    print(f">>> POST {RPC}")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

    t0 = time.time()
    resp = httpx.post(RPC, json=payload, timeout=180.0)
    body = resp.json()
    print(f"\n<<< HTTP {resp.status_code}  ({time.time() - t0:.1f}s)")

    task = body.get("result", {})
    print(f"    kind          {task.get('kind')}")
    print(f"    task id       {task.get('id')}")
    print(f"    contextId     {task.get('contextId')}")
    print(f"    status.state  {(task.get('status') or {}).get('state')}   ← Task 生命週期的終點")
    print(f"    history       {len(task.get('history', []))} 則訊息")
    print(f"    artifacts     {len(task.get('artifacts', []))} 個")

    print("\n    history 逐則（看得到遠端 agent 內部的 tool call）:")
    for m in task.get("history", []):
        for p in m.get("parts", []):
            meta = (p.get("metadata") or {}).get("adk_type", "")
            if p.get("kind") == "text":
                print(f"      [{m['role']:<5}] text: {p['text'][:70]}…")
            elif p.get("kind") == "data":
                d = p.get("data", {})
                print(f"      [{m['role']:<5}] {meta}: {d.get('name')} "
                      f"{json.dumps(d.get('args') or d.get('response'), ensure_ascii=False)[:70]}…")

    answer = task["artifacts"][0]["parts"][0]["text"] if task.get("artifacts") else "(無)"
    print("\n    ✅ 答案（result.artifacts[0].parts[0].text）:")
    print("    " + answer[:400].replace("\n", "\n    "))

    usage = (task.get("metadata") or {}).get("adk_usage_metadata")
    if usage:
        print(f"\n    附帶的 token 用量: {json.dumps(usage, ensure_ascii=False)[:150]}")

In [ ]:
# ── 1.x 原生方言：要帶 A2A-Version header ───────────────────────────────
if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    RPC = A2A_SERVERS["market-analyst"]["rpc"]
    native = {
        "jsonrpc": "2.0", "id": 2, "method": "SendMessage",
        "params": {"message": {"role": "ROLE_USER",
                               "parts": [{"text": "分析 AAPL"}],
                               "messageId": uuid.uuid4().hex}},
    }

    # 先示範「忘記帶 header」會怎樣
    bad = httpx.post(RPC, json=native, timeout=60).json()
    print("❌ 沒帶 A2A-Version header:")
    print("   ", json.dumps(bad.get("error", bad), ensure_ascii=False)[:220])

    # 帶上就正常
    good = httpx.post(RPC, json=native, headers={"A2A-Version": "1.0"}, timeout=60).json()
    task = good.get("result", {}).get("task", {})
    print("\n✅ 帶了 A2A-Version: 1.0:")
    print(f"    result.task.id            {task.get('id')}")
    print(f"    result.task.status.state  {(task.get('status') or {}).get('state')}"
          f"   ← 注意是 TASK_STATE_* 格式")
    print(f"    metadata.adk_author       {(task.get('metadata') or {}).get('adk_author')}")

### 🌊 SSE Streaming

`message/stream` 走 Server-Sent Events，可以看到 Task 一步步推進。

但**要先把 streaming 能力打開**：ADK 自動產生的 AgentCard 是
`capabilities.streaming = false`，而 a2a-sdk 的 handler 會照 card 上寫的擋掉，
於是你會拿到 `-32603 Streaming is not supported by the agent`。

解法是自己用 `AgentCardBuilder` 建一張 card，明確給 `AgentCapabilities(streaming=True)`，
再用 `to_a2a(..., agent_card=card)` 發佈。

In [ ]:
# ── 開了 streaming 的 A2A server ────────────────────────────────────────
from a2a.types import AgentCapabilities

from google.adk.a2a.utils.agent_card_builder import AgentCardBuilder

if "stream-analyst" not in A2A_SERVERS:
    stream_port = free_port()
    stream_card = await AgentCardBuilder(
        agent=market_analyst_wf,
        rpc_url=f"http://127.0.0.1:{stream_port}/",
        capabilities=AgentCapabilities(streaming=True),   # ← 關鍵
        agent_version="2.0.0",
    ).build()

    # serve_a2a 內部會自己挑 port，所以這裡直接用它、把 card 傳進去
    st_info = serve_a2a(market_analyst_wf, "stream-analyst", agent_card=stream_card)
else:
    st_info = A2A_SERVERS["stream-analyst"]

print("streaming capabilities:", json.dumps(st_info["card"]["capabilities"]))
print("（card 上的 url 是我們自己給的，跟實際 port 可能不同，這裡直接用實際的 rpc）")

In [ ]:
if not HAS_KEY:
    print("SKIPPED：沒有 API Key")
else:
    payload = {
        "jsonrpc": "2.0", "id": 3, "method": "message/stream",
        "params": {"message": {"role": "user",
                               "parts": [{"kind": "text", "text": "分析 TSLA"}],
                               "messageId": uuid.uuid4().hex, "kind": "message"}},
    }
    print(f">>> POST {st_info['rpc']}  (method=message/stream)\n")

    with httpx.stream("POST", st_info["rpc"], json=payload, timeout=180.0) as r:
        print(f"<<< HTTP {r.status_code}  content-type={r.headers.get('content-type')}\n")
        for line in r.iter_lines():
            if not line.startswith("data:"):
                continue
            frame = json.loads(line[5:])
            res = frame.get("result") or {}
            if "error" in frame:
                print("   ❌", frame["error"]); break
            kind = res.get("kind")
            state = (res.get("status") or {}).get("state")
            final = res.get("final")
            if kind == "artifact-update":
                text = res["artifact"]["parts"][0].get("text", "")
                print(f"   📦 artifact-update  ({len(text)} 字)  lastChunk={res.get('lastChunk')}")
            else:
                print(f"   ▸ {kind:<16} state={state}" + ("   ← final" if final else ""))

### 🤖 ADK 原生 client：`RemoteA2aAgent`

裸 JSON-RPC 是為了看清協定。實際寫程式時用 `RemoteA2aAgent`：
給它一個 AgentCard 的 URL，它就會自己去 `supportedInterfaces` 找 endpoint、
自己處理 Task 輪詢、把結果轉成 ADK 的 `Event`。

**關鍵是它是 `BaseNode`** —— 所以下一步可以直接把它塞進 `Workflow` 當節點。

In [ ]:
remote_market = RemoteA2aAgent(
    name="remote_market_analyst",
    description="遠端的基本面分析 agent（透過 A2A 呼叫）。",
    agent_card=A2A_SERVERS["market-analyst"]["card_url"],
)
remote_strategist = RemoteA2aAgent(
    name="remote_trading_strategist",
    description="遠端的技術分析與交易策略 agent（透過 A2A 呼叫）。",
    agent_card=A2A_SERVERS["trading-strategist"]["card_url"],
)

print(f"✅ 兩個 RemoteA2aAgent 已建立")
print(f"   都是 ADK BaseNode: "
      f"{isinstance(remote_market, BaseNode)} / {isinstance(remote_strategist, BaseNode)}")

if not HAS_KEY:
    print("\nSKIPPED：沒有 API Key，不實際呼叫")
else:
    runner = InMemoryRunner(agent=remote_market, app_name="a2a_client")
    session = await runner.session_service.create_session(
        app_name="a2a_client", user_id="student")

    print("\n🚀 透過 RemoteA2aAgent 呼叫遠端 agent（走真的 HTTP）\n")
    async for ev in runner.run_async(
        user_id="student", session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text="分析 2330.TW")]),
    ):
        text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else []))
        if text.strip():
            print(f"   📨 author={ev.author}")
            print("   " + text[:400].replace("\n", "\n   "))

### 🎯 重點：A2A Coordinator Workflow

這是整本教材的核心示範。因為 `RemoteA2aAgent` 是 `BaseNode`，
所以**遠端 agent 可以直接當 ADK `Workflow` 的節點**：

```
START
  │  fan-out（兩個遠端呼叫同時發車）
  ├──────────────────┬──────────────────┐
  ▼                  ▼
remote_market   remote_strategist        ← 各自透過 JSON-RPC over HTTP
（ADK Workflow）（LangGraph 橋接）           打到不同的 process
  │                  │
  └────────┬─────────┘  JoinNode（等兩邊到齊）
           ▼
      synthesiser（本地 LlmAgent，把兩份報告寫成投資建議）
```

一段程式碼同時串起了：**ADK Workflow + LangGraph + A2A 協定 + 平行編排**。
而且因為走的是 HTTP，把任一個 agent 搬到 Cloud Run 上，
只要改 AgentCard 的 URL，這段編排一行都不用改。

**`JoinNode` 下游的 `LlmAgent` 會自動收到匯合結果當輸入**，
形狀是 `{"remote_market_analyst": "…", "remote_trading_strategist": "…"}`。
所以 synthesiser 的 instruction 不用寫 state 模板，直接說「你會收到兩份報告」就行。

In [ ]:
# ── 組 Coordinator Workflow ─────────────────────────────────────────────
merged = JoinNode(name="merged")          # 只建一個實例，兩條邊都引用它

# JoinNode 下游的 LlmAgent 會**自動收到匯合結果**當輸入
#   （形狀是 {"remote_market_analyst": "…", "remote_trading_strategist": "…"}），
# 所以 instruction 裡不需要 {merged} 這種 state 模板，直接說「你收到兩份報告」就好。
synthesiser = LlmAgent(
    name="synthesiser",
    model=MODEL,
    description="整合基本面報告與技術面策略，產出最終投資建議。",
    instruction=(
        "你是 StockPulse 的投資決策委員會主席。\n"
        "你會收到兩個專門 agent 透過 A2A 回傳的報告："
        "`remote_market_analyst`（基本面）與 `remote_trading_strategist`（技術面與交易策略）。\n\n"
        "請用繁體中文輸出最終建議，格式：\n"
        "**綜合結論**（100 字內）\n"
        "**兩份報告的共識**（列點）\n"
        "**兩份報告的分歧**（列點；沒有分歧就寫「無明顯分歧」）\n"
        "**最終操作建議**：BUY / SELL / HOLD + 一句理由\n"
        "**風險提醒**（1-2 點）\n"
        "只根據上面兩份報告，不要引入新資訊。"
    ),
)

a2a_coordinator = Workflow(
    name="a2a_coordinator",
    description="fan-out 到兩個遠端 A2A agent，匯合後由本地 LlmAgent 綜合。",
    edges=[
        (START, (remote_market, remote_strategist)),          # fan-out：兩個遠端呼叫同時發車
        ((remote_market, remote_strategist), merged),          # fan-in：JoinNode 等兩邊
        (merged, synthesiser),
    ],
)

coordinator_app = App(
    name="stockpulse_coordinator",
    root_agent=a2a_coordinator,
    resumability_config=ResumabilityConfig(is_resumable=True),
)

print(f"✅ Coordinator Workflow: {a2a_coordinator.name}")
for e in a2a_coordinator.graph.edges:
    print(f"   {e.from_node.name:>28} → {e.to_node.name}")

### 👀 A2A 拓樸圖

同一個 `show_graph()`。圖上的 `remote_market_analyst` / `remote_trading_strategist`
兩個節點，實際上是**跨 process 的 HTTP 呼叫**。

In [ ]:
show_graph(coordinator_app)

In [ ]:
# ── 跑 Coordinator：一次看完跨 process 的 agent 協作 ────────────────────
COORD_TICKER = "NVDA"

coord_events = []
final_advice = ""
a2a_reports = {}

if not HAS_KEY:
    print("SKIPPED：沒有 API Key（graph 已經畫出來了）")
else:
    runner = InMemoryRunner(app=coordinator_app)
    session = await runner.session_service.create_session(
        app_name="stockpulse_coordinator", user_id="student")

    print(f"🚀 A2A Coordinator：分析 {COORD_TICKER}")
    print("   （兩個遠端呼叫同時發車，所以總時間 ≈ 較慢的那一個，不是兩者相加）\n")
    t0 = time.time()

    async for ev in runner.run_async(
        user_id="student", session_id=session.id,
        new_message=types.Content(role="user",
                                  parts=[types.Part(text=f"分析 {COORD_TICKER}")]),
    ):
        coord_events.append(ev)
        path = getattr(getattr(ev, "node_info", None), "path", None)
        node_name = path.split("/")[-1].split("@")[0] if path else None
        text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else "")).strip()

        if ev.output is not None:              # JoinNode 的匯合結果
            keys = list(ev.output) if isinstance(ev.output, dict) else ev.output
            print(f"   🔀 [{time.time() - t0:5.1f}s] {node_name:<26} 匯合了 {keys}")
        elif text:
            print(f"   📨 [{time.time() - t0:5.1f}s] {node_name:<26} 回傳 {len(text)} 字")
            if node_name == "synthesiser":
                final_advice = text
            elif node_name:
                a2a_reports[node_name] = text

    print(f"\n✅ 完成，總耗時 {time.time() - t0:.1f}s，共 {len(coord_events)} 個 event")

In [ ]:
# ── 上色版的拓樸圖 ──────────────────────────────────────────────────────
if coord_events:
    show_graph(coordinator_app, agent_state=node_status_from_events(coord_events))
else:
    print("SKIPPED：上面沒有跑")

In [ ]:
# ── 最終報告 ─────────────────────────────────────────────────────────────
if not final_advice:
    print("SKIPPED：沒有結果")
else:
    sections = "\n".join(
        f"#### 📨 {name}（透過 A2A）\n\n{text}\n" for name, text in a2a_reports.items())
    display(Markdown(f"""## 📊 StockPulse AI 完整分析報告

**股票代碼：** {COORD_TICKER}
**分析時間：** {pd.Timestamp.now():%Y-%m-%d %H:%M:%S}
**編排方式：** ADK 2.0 `Workflow`，兩個節點是走 JSON-RPC over HTTP 的遠端 A2A agent

---

{sections}

---

### 🎯 綜合投資建議（本地 synthesiser LlmAgent）

{final_advice}

---

### 🔗 A2A 連線狀態

| Agent | 傳輸 | Endpoint | 狀態 |
|---|---|---|---|
| `remote_market_analyst` | JSON-RPC / HTTP | `{A2A_SERVERS['market-analyst']['rpc']}` | ✅ |
| `remote_trading_strategist` | JSON-RPC / HTTP | `{A2A_SERVERS['trading-strategist']['rpc']}` | ✅ |
| `synthesiser` | 本地 | — | ✅ |

> ⚠️ 以上內容由 AI 產生，僅供教學示範，不構成投資建議。
"""))

# 6️⃣ FastAPI Gateway

把整套東西包成一個可部署的 HTTP 服務。這裡示範 ADK 2.0 的一個好用之處：
**`to_a2a()` 回傳的是標準 Starlette app，可以直接 `mount` 進 FastAPI**。

於是一個 Cloud Run service 就同時提供：

| 路徑 | 內容 |
|---|---|
| `/api/data/*` | yfinance 資料代理 |
| `/api/analyze/{ticker}` | 走完整 A2A Coordinator |
| `/a2a/market-analyst/*` | mount 進來的 A2A server（含 AgentCard） |
| `/a2a/trading-strategist/*` | 同上 |
| `/docs` | 自動產生的 Swagger |

這樣一來 A2A 是「真的 HTTP」，只是 host 剛好是 localhost；
未來要拆成多個服務，只要改 AgentCard 的 URL。

In [ ]:
# ── 先決定 gateway 的 port，因為 AgentCard 上要寫對外的 URL ───────────────
# to_a2a() 內部會用 host/port 組出 rpc_url 寫進 card。但 mount 之後對外路徑是
# /a2a/<key>/，跟 to_a2a 自己算的不一樣，所以我們自己建 card 把正確 URL 寫進去。
API_SERVERS = globals().setdefault("API_SERVERS", {})
GW_PORT = API_SERVERS["api"]["port"] if "api" in API_SERVERS else free_port()
GW_BASE = f"http://127.0.0.1:{GW_PORT}"

MOUNTED = {"market-analyst": market_analyst_wf,
           "trading-strategist": trading_strategist}

mounted_cards = {}
for key, agent in MOUNTED.items():
    mounted_cards[key] = await AgentCardBuilder(
        agent=agent,
        rpc_url=f"{GW_BASE}/a2a/{key}/",     # ← 對外看到的位址
        agent_version="2.0.0",
    ).build()

print(f"gateway port  {GW_PORT}")
for key, card in mounted_cards.items():
    print(f"  {key:<20} card.url → {GW_BASE}/a2a/{key}/")

In [ ]:
# ── 建 FastAPI app ───────────────────────────────────────────────────────
from contextlib import AsyncExitStack, asynccontextmanager

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware

# to_a2a() 產生的 Starlette sub-app
a2a_subapps = {key: to_a2a(agent, host="127.0.0.1", port=GW_PORT,
                           agent_card=mounted_cards[key])
               for key, agent in MOUNTED.items()}


# ⚠️ 這一段是 mount A2A app 的關鍵。
#    to_a2a() 是在 **Starlette lifespan** 裡才把 A2A 路由掛上去的
#    （去看 google/adk/a2a/utils/agent_to_a2a.py 的 setup_a2a）。
#    而 Starlette 的 Mount **不會**自動跑 sub-app 的 lifespan，
#    所以直接 api.mount(...) 的話，那些路由永遠不會存在 → AgentCard 拿到 404。
#    正解：在父 app 的 lifespan 裡，手動進入每個 sub-app 的 lifespan context。
@asynccontextmanager
async def gateway_lifespan(app: FastAPI):
    async with AsyncExitStack() as stack:
        for sub in a2a_subapps.values():
            await stack.enter_async_context(sub.router.lifespan_context(sub))
        yield


api = FastAPI(
    title="StockPulse AI API",
    description="股市智慧儀表板 — ADK 2.0 + A2A",
    version="2.0.0",
    lifespan=gateway_lifespan,
)
api.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True,
                   allow_methods=["*"], allow_headers=["*"])


@api.get("/")
def root():
    return {"service": "StockPulse AI", "version": "2.0.0", "adk": md.version("google-adk"),
            "timestamp": pd.Timestamp.now().isoformat()}


@api.get("/api/health")
def health():
    return {"status": "healthy",
            "standalone_a2a": {k: v["base"] for k, v in A2A_SERVERS.items()},
            "mounted_a2a": [f"/a2a/{k}/" for k in a2a_subapps]}


@api.get("/api/data/quote/{ticker}")
def api_quote(ticker: str):
    data = get_stock_info(ticker)
    if "error" in data:
        raise HTTPException(404, data["error"])
    return data


@api.get("/api/data/history/{ticker}")
def api_history(ticker: str, period: str = "3mo"):
    data = get_stock_history(ticker, period)
    if "error" in data:
        raise HTTPException(404, data["error"])
    return data


@api.get("/api/data/indicators/{ticker}")
def api_indicators(ticker: str, period: str = "6mo"):
    data = calculate_technical_indicators(ticker, period)
    if "error" in data:
        raise HTTPException(404, data["error"])
    return data


@api.get("/api/data/news/{ticker}")
def api_news(ticker: str, top_n: int = 5):
    return get_market_news(ticker, top_n)


@api.get("/api/agents")
def api_agents():
    """列出所有 A2A agent 的 card（服務發現用）。"""
    return {"mounted": [f"/a2a/{k}{AGENT_CARD_WELL_KNOWN_PATH}" for k in a2a_subapps],
            "standalone": {k: v["card"] for k, v in A2A_SERVERS.items()}}


@api.get("/api/analyze/{ticker}")
async def api_analyze(ticker: str):
    """跑完整的 A2A Coordinator Workflow。"""
    if not HAS_KEY:
        raise HTTPException(503, "伺服器沒有設定 GOOGLE_API_KEY")

    runner = InMemoryRunner(app=coordinator_app)
    session = await runner.session_service.create_session(
        app_name="stockpulse_coordinator", user_id="api")

    reports, advice = {}, ""
    async for ev in runner.run_async(
        user_id="api", session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=f"分析 {ticker}")]),
    ):
        path = getattr(getattr(ev, "node_info", None), "path", None)
        name = path.split("/")[-1].split("@")[0] if path else None
        text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else "")).strip()
        if text and name == "synthesiser":
            advice = text
        elif text and name:
            reports[name] = text

    return {"ticker": ticker, "timestamp": pd.Timestamp.now().isoformat(),
            "agent_reports": reports, "final_advice": advice}


# ⬇️ 把 A2A Starlette app mount 進來（路由靠上面的 gateway_lifespan 才會生效）
for key, sub in a2a_subapps.items():
    api.mount(f"/a2a/{key}", sub)

print("✅ FastAPI app 已建立")
print("   REST : /api/health /api/data/{quote,history,indicators,news}/{ticker}")
print("          /api/agents /api/analyze/{ticker}")
print("   A2A  : " + "  ".join(f"/a2a/{k}/" for k in a2a_subapps))
print("   Docs : /docs")

In [ ]:
# ── 起 server 並測試 ─────────────────────────────────────────────────────
if "api" not in API_SERVERS:
    srv = uvicorn.Server(uvicorn.Config(api, host="127.0.0.1", port=GW_PORT,
                                        log_level="error"))
    srv.install_signal_handlers = lambda: None
    thread = threading.Thread(target=srv.run, daemon=True, name="fastapi")
    thread.start()
    deadline = time.time() + 30
    while not srv.started and time.time() < deadline:
        if not thread.is_alive():
            raise RuntimeError("FastAPI 的 uvicorn thread 死了")
        time.sleep(0.05)
    if not srv.started:
        raise TimeoutError("FastAPI 沒有在 30s 內啟動")
    API_SERVERS["api"] = {"srv": srv, "thread": thread, "port": GW_PORT, "base": GW_BASE}

print(f"🌐 FastAPI 已啟動: {GW_BASE}")
print(f"   Swagger: {GW_BASE}/docs\n")

print("── REST 端點 ──")
for path in ["/", "/api/health", "/api/data/quote/NVDA",
             "/api/data/indicators/2330.TW", "/api/data/news/NVDA?top_n=1"]:
    r = httpx.get(GW_BASE + path, timeout=60)
    body = json.dumps(r.json(), ensure_ascii=False, default=str)
    print(f"GET {path:<38} → {r.status_code}  {body[:100]}…")

print("\n── mount 進來的 A2A（AgentCard + 真的 JSON-RPC）──")
for key in a2a_subapps:
    r = httpx.get(f"{GW_BASE}/a2a/{key}{AGENT_CARD_WELL_KNOWN_PATH}", timeout=30)
    if r.status_code != 200:
        print(f"GET /a2a/{key}{AGENT_CARD_WELL_KNOWN_PATH} → {r.status_code} ❌")
        continue
    card = r.json()
    print(f"GET /a2a/{key}{AGENT_CARD_WELL_KNOWN_PATH} → 200  "
          f"name={card['name']}  skills={len(card['skills'])}")
    print(f"     card 對外 url: {card['supportedInterfaces'][0]['url']}")

if HAS_KEY:
    r = httpx.post(f"{GW_BASE}/a2a/market-analyst/", timeout=180, json={
        "jsonrpc": "2.0", "id": 9, "method": "message/send",
        "params": {"message": {"role": "user",
                               "parts": [{"kind": "text", "text": "分析 AAPL"}],
                               "messageId": uuid.uuid4().hex, "kind": "message"}}})
    task = r.json().get("result", {})
    ans = task["artifacts"][0]["parts"][0]["text"] if task.get("artifacts") else "(無)"
    print(f"\nPOST /a2a/market-analyst/  → {r.status_code}  "
          f"state={(task.get('status') or {}).get('state')}")
    print(f"     {ans[:180]}…")
else:
    print("\n（沒有 API Key，跳過 JSON-RPC 呼叫）")

# 7️⃣ Dashboard 預覽

用 `IPython.display.HTML` 直接在 notebook 裡渲染一個儀表板。
實務上這一段會換成 React 前端打 §6 的 REST API。

In [ ]:
from IPython.display import HTML

DASH_TICKER = "NVDA"
quote = get_stock_info(DASH_TICKER)
ind = calculate_technical_indicators(DASH_TICKER, "6mo")
hist = get_stock_history(DASH_TICKER, "3mo")
news = get_market_news(DASH_TICKER, top_n=4)

if "error" in quote or "error" in ind:
    print("⚠️ 抓不到資料，跳過 dashboard")
else:
    rsi = ind["rsi"]
    rsi_color = "#ef4444" if rsi > 70 else "#34d399" if rsi < 30 else "#fbbf24"
    trend_color = "#34d399" if "多頭" in ind["ma_trend"] else "#ef4444"
    chg = quote["current_price"] - quote["previous_close"]
    chg_pct = (chg / quote["previous_close"] * 100) if quote["previous_close"] else 0
    chg_color = "#34d399" if chg >= 0 else "#ef4444"

    def card(label, value, sub="", color="#e2e8f0"):
        return f"""<div style="background:rgba(255,255,255,.05);border:1px solid rgba(255,255,255,.1);
          border-radius:12px;padding:16px">
          <div style="color:#94a3b8;font-size:12px;letter-spacing:.5px">{label}</div>
          <div style="color:{color};font-size:24px;font-weight:700;margin-top:6px">{value}</div>
          <div style="color:#64748b;font-size:11px;margin-top:4px">{sub}</div></div>"""

    news_html = "".join(
        f"""<div style="padding:10px 0;border-bottom:1px solid rgba(255,255,255,.07)">
        <div style="color:#e2e8f0;font-size:13px;line-height:1.45">{n['title'][:110]}</div>
        <div style="color:#64748b;font-size:11px;margin-top:3px">{n['publisher']}</div></div>"""
        for n in news["items"]) or '<div style="color:#64748b">（無新聞）</div>'

    display(HTML(f"""
<div style="font-family:-apple-system,'Segoe UI',system-ui,sans-serif;max-width:1100px;margin:0 auto;
     background:linear-gradient(135deg,#0f172a,#1e293b);color:#e2e8f0;padding:26px;border-radius:16px;
     box-shadow:0 10px 40px rgba(0,0,0,.35)">

  <div style="display:flex;justify-content:space-between;align-items:flex-end;flex-wrap:wrap;gap:12px">
    <div>
      <div style="font-size:26px;font-weight:800;background:linear-gradient(135deg,#60a5fa,#34d399);
           -webkit-background-clip:text;-webkit-text-fill-color:transparent;background-clip:text">
        🚀 StockPulse AI Dashboard</div>
      <div style="color:#94a3b8;font-size:13px;margin-top:4px">
        ADK 2.0 Workflow · A2A Protocol · LangGraph</div>
    </div>
    <div style="text-align:right">
      <div style="font-size:13px;color:#94a3b8">{quote['company_name'][:34]}</div>
      <div style="font-size:32px;font-weight:800">{quote['current_price']}
        <span style="font-size:14px;color:#94a3b8">{quote['currency']}</span></div>
      <div style="color:{chg_color};font-size:14px;font-weight:600">
        {'▲' if chg >= 0 else '▼'} {abs(chg):.2f} ({chg_pct:+.2f}%)</div>
    </div>
  </div>

  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(155px,1fr));gap:12px;margin:22px 0">
    {card("RSI (14)", f"{rsi}", ind["rsi_signal"], rsi_color)}
    {card("MA 趨勢", ind["ma_trend"], f"MA20 {ind['ma20']} / MA50 {ind['ma50']}", trend_color)}
    {card("MACD", ind["macd_crossover"], f"{ind['macd']} vs {ind['macd_signal']}")}
    {card("布林帶", ind["bb_position"], f"{ind['bb_lower']} ~ {ind['bb_upper']}")}
    {card("本益比", f"{quote['pe_ratio']}", f"股利率 {quote['dividend_yield_pct']}%")}
    {card("3個月漲跌", f"{hist.get('price_change_pct', 0):+.2f}%",
          f"{hist.get('start_price')} → {hist.get('end_price')}",
          "#34d399" if hist.get("price_change_pct", 0) >= 0 else "#ef4444")}
    {card("52週區間", f"{quote['week52_low']} ~ {quote['week52_high']}",
          f"現價位置 {(quote['current_price'] - quote['week52_low']) / max(quote['week52_high'] - quote['week52_low'], 1e-9) * 100:.0f}%")}
    {card("成交量", f"{quote['volume']:,}", f"均量 {quote['average_volume']:,}")}
  </div>

  <div style="display:grid;grid-template-columns:1.15fr 1fr;gap:18px">
    <div style="background:rgba(255,255,255,.04);border-radius:12px;padding:16px">
      <div style="font-size:14px;font-weight:700;margin-bottom:8px">🤖 AI 綜合建議</div>
      <div style="color:#cbd5e1;font-size:12.5px;line-height:1.65;white-space:pre-wrap;
           max-height:270px;overflow-y:auto">{(final_advice or market_report
             or "（尚未執行 AI 分析，請先跑 §3 或 §5）")[:1400]}</div>
    </div>
    <div style="background:rgba(255,255,255,.04);border-radius:12px;padding:16px">
      <div style="font-size:14px;font-weight:700;margin-bottom:4px">📰 相關新聞</div>
      <div style="max-height:270px;overflow-y:auto">{news_html}</div>
    </div>
  </div>

  <div style="margin-top:18px;padding-top:12px;border-top:1px solid rgba(255,255,255,.08);
       color:#64748b;font-size:11px;display:flex;justify-content:space-between;flex-wrap:wrap;gap:8px">
    <span>資料：Yahoo Finance · 分析：Gemini {MODEL}</span>
    <span>{pd.Timestamp.now():%Y-%m-%d %H:%M:%S} · ⚠️ 僅供教學，不構成投資建議</span>
  </div>
</div>"""))

# 8️⃣ Podcast 觀點融合模組（選配）

## 為什麼要這個

AI Agent 很擅長處理數字和技術指標，但**人類專家的產業人脈、對未公開資訊的判讀、
以及那種「我覺得這波不對」的直覺**，是純資料分析拿不到的。
這個模組把財經 Podcast 的觀點結構化之後，注入 §5 的 Coordinator，
讓 Trading Strategist 去**批判性地**融合這些觀點，而不是照抄。

## 流程（Human-in-the-loop）

```
使用者貼 YouTube URL
      ↓  yt-dlp 下載音訊
      ↓  Gemini 直接聽音訊 → 轉錄 + 結構化摘要（JSON）
      ↓  人工審核 / 編輯          ← 這一步不能省，AI 會聽錯代號
      ↓  確認後存成 podcast_insights/*.json
      ↓  依 ticker 篩選相關觀點 → 注入 A2A Coordinator 的 query
```

## ⚠️ 這一節預設不會執行

`RUN_PODCAST = False`。因為它需要：

- 額外安裝 `yt-dlp`（`pip install yt-dlp`）
- 從 YouTube 下載音訊（會受 YouTube 限流影響，不保證每次成功）
- 上傳音訊到 Gemini 並等它聽完（一集 60 分鐘的 podcast 要好幾分鐘）

**函式都會被定義好**（所以整本 notebook 從頭跑到尾不會斷），
想真的跑的話把 `RUN_PODCAST` 改成 `True` 並填入 `PODCAST_URL`。

In [ ]:
# ── Podcast 模組開關 ────────────────────────────────────────────────────
RUN_PODCAST = False                                    # ← 想真的跑就改 True
PODCAST_URL = "https://www.youtube.com/watch?v=XXXXXXXXXXX"   # ← 換成真的連結
PODCAST_TICKERS = ["2330.TW", "2454.TW"]               # 重點關注的股票

# 安裝：!pip install -q yt-dlp
import hashlib
import re
import shutil
import subprocess
from pathlib import Path

PODCAST_AUDIO_DIR = Path("podcast_audio")
PODCAST_INSIGHTS_DIR = Path("podcast_insights")
PODCAST_AUDIO_DIR.mkdir(exist_ok=True)
PODCAST_INSIGHTS_DIR.mkdir(exist_ok=True)


def download_audio(url: str) -> dict:
    """用 yt-dlp 下載音訊並抓 metadata。

    Args:
        url: YouTube 連結，或 mp3 直連 URL。
    Returns:
        {"audio_path", "title", "upload_date", "channel", "url"}
    """
    if shutil.which("yt-dlp") is None:
        raise RuntimeError("找不到 yt-dlp，請先 `pip install yt-dlp`")

    if "youtube.com" in url or "youtu.be" in url:
        dl = subprocess.run(
            ["yt-dlp", "--extract-audio", "--audio-format", "mp3", "--no-playlist",
             "--print", "after_move:filepath",
             "--output", str(PODCAST_AUDIO_DIR / "%(title)s.%(ext)s"), url],
            capture_output=True, text=True, timeout=1800)
        if dl.returncode != 0:
            raise RuntimeError(f"yt-dlp 下載失敗: {dl.stderr[-400:]}")
        audio_path = dl.stdout.strip().splitlines()[-1]

        meta = subprocess.run(
            ["yt-dlp", "--print", "%(title)s|||%(upload_date>%Y-%m-%d)s|||%(channel)s",
             "--no-playlist", url],
            capture_output=True, text=True, timeout=120).stdout.strip().split("|||")
        meta += [""] * (3 - len(meta))
        return {"audio_path": audio_path, "title": meta[0] or "Unknown",
                "upload_date": meta[1], "channel": meta[2] or "Unknown", "url": url}

    # 直接 mp3 連結
    dest = PODCAST_AUDIO_DIR / (hashlib.md5(url.encode()).hexdigest()[:12] + ".mp3")
    if not dest.exists():
        with httpx.stream("GET", url, timeout=600, follow_redirects=True) as r:
            r.raise_for_status()
            with open(dest, "wb") as f:
                for chunk in r.iter_bytes():
                    f.write(chunk)
    return {"audio_path": str(dest), "title": dest.stem, "upload_date": "",
            "channel": "RSS", "url": url}


print("✅ download_audio() 已定義")
print(f"   yt-dlp: {shutil.which('yt-dlp') or '❌ 未安裝（RUN_PODCAST=True 時才需要）'}")

In [ ]:
def transcribe_and_summarize(audio_info: dict, target_tickers: list | None = None) -> dict:
    """把音訊丟給 Gemini 直接聽，轉錄並輸出結構化摘要。

    Gemini 是原生多模態，可以直接吃音訊檔，不需要先跑 Whisper 之類的 ASR。

    Args:
        audio_info: download_audio() 的回傳值。
        target_tickers: 重點關注的股票代號清單（會寫進 prompt 提醒模型別漏）。
    Returns:
        含 episode / podcast_name / date / insights 的 dict。
    """
    from google import genai
    from google.genai.types import GenerateContentConfig

    client = genai.Client(api_key=GOOGLE_API_KEY)
    src = Path(audio_info["audio_path"])

    # Gemini SDK 對非 ASCII 檔名不友善，先複製成 hash 檔名
    safe = src.parent / (hashlib.md5(src.name.encode()).hexdigest()[:12] + src.suffix)
    if not safe.exists():
        shutil.copy2(src, safe)

    print(f"📋 {audio_info['title']}")
    print(f"   {audio_info['channel']} | {audio_info['upload_date']}")
    print("⏳ 上傳音訊到 Gemini…")
    audio_file = client.files.upload(file=str(safe))

    focus = ""
    if target_tickers:
        focus = f"\n【重點關注】使用者特別關注：{', '.join(target_tickers)}，這些股票的討論請完整提取。\n"

    prompt = f"""你是專業的台灣股市分析助理，正在處理財經 Podcast 音訊。

【已知資訊】標題「{audio_info['title']}」／頻道「{audio_info['channel']}」／日期 {audio_info['upload_date']}
請直接用上面的已知資訊填 episode、podcast_name、date，不要自己猜。

請聽完整段音訊，提取結構化資訊：
1. 所有提到的股票要轉成交易所格式（台積電 → 2330.TW）
2. 每個觀點分類為 stock（個股）/ strategy（心法）/ sector（產業）/ macro（總經）
3. stock 與 sector 要判斷情緒 bullish / bearish / neutral
4. 全部用繁體中文
5. 有具體價位或進出場時機要記在 content 裡
6. 風險提示在 key_points 用「⚠️ 風險:」開頭

嚴格輸出這個 JSON 結構：
{{"episode":"節目名 EP.XXX","podcast_name":"節目名","date":"YYYY-MM-DD","duration_minutes":0,
  "insights":[{{"type":"stock","tickers":["2330.TW"],"sentiment":"bullish",
                "content":"摘要","key_points":["論點1"]}}]}}

保持客觀，忠實呈現 Podcast 的觀點，不要加入自己的看法。只輸出 JSON。
{focus}"""

    print("⏳ Gemini 正在聽（一集 60 分鐘大約要 2-5 分鐘）…")
    resp = client.models.generate_content(
        model=MODEL,
        contents=[audio_file, prompt],
        config=GenerateContentConfig(response_mime_type="application/json", temperature=0.3),
    )

    text = (resp.text or "").strip()
    if text.startswith("```"):                    # 保險：把 markdown code fence 拆掉
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0]
    try:
        summary = json.loads(text)
    except json.JSONDecodeError:
        print("⚠️ JSON 被截斷，請 Gemini 修一次…")
        fix = client.models.generate_content(
            model=MODEL,
            contents=[f"以下是被截斷的 JSON，請修復補全，只輸出合法 JSON：\n\n{text[:12000]}"],
            config=GenerateContentConfig(response_mime_type="application/json", temperature=0.1),
        )
        summary = json.loads((fix.text or "{}").strip())

    tickers = {t for ins in summary.get("insights", []) for t in ins.get("tickers", [])}
    print(f"✅ 完成：{len(summary.get('insights', []))} 條觀點，涉及 {len(tickers)} 支股票 "
          f"{', '.join(sorted(tickers))}")
    return summary


print("✅ transcribe_and_summarize() 已定義")

In [ ]:
def review_and_confirm(summary: dict) -> dict:
    """把摘要排版顯示出來給人審核（Human-in-the-loop 的關鍵一步）。"""
    type_emoji = {"stock": "📊", "strategy": "🧠", "sector": "🏭", "macro": "🌍"}
    sent_emoji = {"bullish": "🟢 看多", "bearish": "🔴 看空", "neutral": "🟡 中立"}

    out = [f"## 🎙️ Podcast 摘要審核\n",
           f"**節目：** {summary.get('episode', 'N/A')}  ",
           f"**Podcast：** {summary.get('podcast_name', 'N/A')}  ",
           f"**日期：** {summary.get('date', 'N/A')}  ",
           f"**長度：** {summary.get('duration_minutes', 'N/A')} 分鐘\n", "---\n"]

    for i, ins in enumerate(summary.get("insights", []), 1):
        out.append(f"### {type_emoji.get(ins.get('type'), '💬')} 觀點 {i} "
                   f"[{str(ins.get('type', '')).upper()}]\n")
        if ins.get("tickers"):
            out.append(f"**相關股票：** {', '.join(ins['tickers'])}  ")
        if ins.get("sentiment"):
            out.append(f"**情緒：** {sent_emoji.get(ins['sentiment'], ins['sentiment'])}  ")
        out.append(f"\n{ins.get('content', '')}\n")
        for kp in ins.get("key_points", []):
            out.append(f"- {kp}")
        out.append("\n---\n")

    display(Markdown("\n".join(out)))
    return summary


def confirm_and_save(summary: dict) -> dict:
    """審核通過後存檔。存檔前可以先直接改 summary dict 修正 AI 聽錯的地方。"""
    summary["status"] = "confirmed"
    summary["processed_at"] = pd.Timestamp.now().isoformat()

    date = summary.get("date") or pd.Timestamp.now().strftime("%Y-%m-%d")
    fname = re.sub(r"[^\w\-.]", "_",
                   f"{date}_{summary.get('podcast_name', 'unknown')}_"
                   f"{summary.get('episode', '')}") + ".json"
    path = PODCAST_INSIGHTS_DIR / fname
    path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"✅ 已存檔: {path}")
    return summary


def filter_relevant_insights(target_ticker: str) -> list:
    """從存檔的觀點裡挑出跟目標股票相關的。

    篩選邏輯（OR）：tickers 含目標代號，或 type == "strategy"（通用心法適用所有股票）。
    """
    relevant = []
    for fp in sorted(PODCAST_INSIGHTS_DIR.glob("*.json")):
        data = json.loads(fp.read_text(encoding="utf-8"))
        if data.get("status") != "confirmed":
            continue
        for ins in data.get("insights", []):
            if target_ticker in ins.get("tickers", []) or ins.get("type") == "strategy":
                relevant.append({**ins,
                                 "source_episode": data.get("episode", ""),
                                 "source_date": data.get("date", "")})
    print(f"📋 找到 {len(relevant)} 條與 {target_ticker} 相關的 Podcast 觀點")
    return relevant


def format_podcast_context(insights: list) -> str:
    """把觀點排成可以塞進 agent query 的文字。"""
    if not insights:
        return ""
    lines = ["【Podcast 專家觀點參考（人類觀點，請批判性採用，不要照抄）】\n"]
    for i, ins in enumerate(insights, 1):
        sent = f" [{ins.get('sentiment')}]" if ins.get("sentiment") else ""
        lines.append(f"{i}. [{str(ins.get('type', '')).upper()}]{sent} "
                     f"({ins.get('source_episode', 'N/A')} - {ins.get('source_date', '')})")
        lines.append(f"   {ins.get('content', '')}")
        for kp in ins.get("key_points", []):
            lines.append(f"   - {kp}")
        lines.append("")
    return "\n".join(lines)


print("✅ review_and_confirm / confirm_and_save / filter_relevant_insights / "
      "format_podcast_context 已定義")

In [ ]:
def cross_podcast_summary(episodes: list) -> dict:
    """跨多集彙整：找出共識看多／看空、觀點分歧、熱門產業、交易心法、風險警示。"""
    if not episodes:
        print("⚠️ 沒有已確認的 Podcast 資料")
        return {}

    from google import genai
    from google.genai.types import GenerateContentConfig

    chunks = []
    for ep in episodes:
        chunks.append(f"\n【{ep.get('podcast_name', '?')} - {ep.get('episode', '?')} "
                      f"({ep.get('date', '?')})】")
        for ins in ep.get("insights", []):
            chunks.append(f"[{ins.get('type')}] {ins.get('sentiment', '')} "
                          f"({', '.join(ins.get('tickers', [])) or '無特定股票'}): "
                          f"{ins.get('content', '')}")
            chunks += [f"  - {kp}" for kp in ins.get("key_points", [])]

    text = "\n".join(chunks)[:100000]
    print(f"📝 彙整 {len(episodes)} 集、共 {len(text)} 字送入 Gemini…")

    prompt = f"""你是資深的台灣股市研究總監。以下是多集財經 Podcast 的結構化摘要。

請做跨集深度彙整，輸出這個 JSON：
{{"analysis_date":"YYYY-MM-DD","episodes_analyzed":0,
  "consensus_bullish":[{{"ticker":"2330.TW","name":"台積電","mentioned_by":["股癌"],
                         "consensus_reason":"理由","confidence":"high"}}],
  "consensus_bearish":[],
  "divergent_opinions":[{{"ticker":"代號","bullish_by":[],"bearish_by":[],"summary":"分歧原因"}}],
  "hot_sectors":[{{"sector":"產業","sentiment":"bullish","related_tickers":[],"summary":"摘要"}}],
  "trading_wisdom":[{{"category":"心法","content":"內容","source":"來源"}}],
  "risk_warnings":["風險"],
  "executive_summary":"200字內總結"}}

規則：
- 只有 2 集以上提到才算「共識」，1 集獨有的放 hot_sectors 或不列
- divergent_opinions 只列真的有人看多有人看空的
- trading_wisdom 相似的合併去重
- 代號統一用台股格式（2330.TW）或美股格式（NVDA）
- 全部繁體中文

資料：
{text}"""

    client = genai.Client(api_key=GOOGLE_API_KEY)
    resp = client.models.generate_content(
        model=MODEL, contents=[prompt],
        config=GenerateContentConfig(response_mime_type="application/json", temperature=0.3))
    result = json.loads((resp.text or "{}").strip())

    print(f"✅ 🟢共識看多 {len(result.get('consensus_bullish', []))} / "
          f"🔴共識看空 {len(result.get('consensus_bearish', []))} / "
          f"⚡分歧 {len(result.get('divergent_opinions', []))} / "
          f"🏭產業 {len(result.get('hot_sectors', []))} / "
          f"🧠心法 {len(result.get('trading_wisdom', []))}")
    return result


def load_all_podcast_insights() -> list:
    """載入所有已確認的 Podcast JSON。"""
    eps = [json.loads(fp.read_text(encoding="utf-8"))
           for fp in sorted(PODCAST_INSIGHTS_DIR.glob("*.json"))]
    eps = [e for e in eps if e.get("status") == "confirmed"]
    total = sum(len(e.get("insights", [])) for e in eps)
    print(f"📂 已載入 {len(eps)} 集、共 {total} 條觀點")
    return eps


def display_cross_summary(summary: dict):
    """排版顯示跨集彙整結果。"""
    if not summary:
        print("⚠️ 無資料")
        return

    out = [f"# 🧠 Podcast 跨集觀點彙整\n",
           f"**分析日期：** {summary.get('analysis_date', 'N/A')}　"
           f"**集數：** {summary.get('episodes_analyzed', 0)}\n",
           "---\n", f"## 📝 總結\n\n{summary.get('executive_summary', 'N/A')}\n"]

    for title, key, extra in [("## 🟢 共識看多", "consensus_bullish", True),
                              ("## 🔴 共識看空", "consensus_bearish", True)]:
        if summary.get(key):
            out.append(f"\n---\n\n{title}\n")
            for it in summary[key]:
                out.append(f"### {it.get('ticker', '')} {it.get('name', '')}")
                out.append(f"- **來源：** {', '.join(it.get('mentioned_by', []))}")
                if it.get("confidence"):
                    out.append(f"- **信心度：** {it['confidence']}")
                out.append(f"- **理由：** {it.get('consensus_reason', '')}\n")

    if summary.get("divergent_opinions"):
        out.append("\n---\n\n## ⚡ 觀點分歧\n")
        for it in summary["divergent_opinions"]:
            out.append(f"### {it.get('ticker', '')}")
            out.append(f"- 🟢 看多：{', '.join(it.get('bullish_by', []))}")
            out.append(f"- 🔴 看空：{', '.join(it.get('bearish_by', []))}")
            out.append(f"- **分歧原因：** {it.get('summary', '')}\n")

    if summary.get("hot_sectors"):
        out.append("\n---\n\n## 🏭 熱門產業\n")
        for it in summary["hot_sectors"]:
            out.append(f"- **{it.get('sector', '')}**（{it.get('sentiment', '')}）"
                       f"{', '.join(it.get('related_tickers', []))} — {it.get('summary', '')}")

    if summary.get("trading_wisdom"):
        out.append("\n---\n\n## 🧠 交易心法\n")
        for it in summary["trading_wisdom"]:
            out.append(f"- **[{it.get('category', '')}]** {it.get('content', '')} "
                       f"_（{it.get('source', '')}）_")

    if summary.get("risk_warnings"):
        out.append("\n---\n\n## ⚠️ 風險警示\n")
        out += [f"- {w}" for w in summary["risk_warnings"]]

    display(Markdown("\n".join(out)))


print("✅ cross_podcast_summary / load_all_podcast_insights / display_cross_summary 已定義")

In [ ]:
# ── 實際執行（預設關閉）─────────────────────────────────────────────────
if not RUN_PODCAST:
    print("⏭  RUN_PODCAST = False，跳過 Podcast 處理。")
    print("   函式都已定義，想真的跑的話：")
    print("     1. pip install yt-dlp")
    print("     2. 把上面的 RUN_PODCAST 改成 True、PODCAST_URL 換成真的連結")
    print("     3. 從這個 cell 重新執行")
    podcast_ctx = ""
elif not HAS_KEY:
    print("⏭  沒有 API Key，跳過")
    podcast_ctx = ""
else:
    # Step 1-3：下載 → Gemini 轉錄摘要 → 人工審核
    audio_info = download_audio(PODCAST_URL)
    summary = transcribe_and_summarize(audio_info, target_tickers=PODCAST_TICKERS)
    review_and_confirm(summary)
    print("\n👀 請審核上面的摘要。要修正的話直接改 summary dict，例如：")
    print('   summary["insights"][0]["sentiment"] = "bearish"')
    print("   確認無誤後執行下一個 cell 存檔。")
    podcast_ctx = ""

In [ ]:
# ── 審核完才執行這個 cell：存檔 + 產生注入用的 context ─────────────────
if not RUN_PODCAST or not HAS_KEY:
    print("⏭  跳過")
    podcast_ctx = ""
else:
    confirm_and_save(summary)                                   # noqa: F821
    relevant = filter_relevant_insights(PODCAST_TICKERS[0])
    podcast_ctx = format_podcast_context(relevant)
    print(f"\n📝 注入用的 context（{len(podcast_ctx)} 字）預覽：")
    print(podcast_ctx[:600])

### 🔗 把 Podcast 觀點注入 A2A Coordinator

融合的做法很簡單：**把觀點附在使用者 query 後面**。
`remote_trading_strategist` 收到的訊息會同時包含「請分析 XXXX」和「Podcast 專家觀點」，
LangGraph 內部的 `strategy_analysis_node` 就會把它一起考慮進去。

注意 prompt 裡寫的是「**批判性採用，不要照抄**」——
Podcast 是人的主觀觀點，agent 應該把它當成一個資料點，而不是答案。

In [ ]:
async def analyze_with_podcast(ticker: str, podcast_context: str = "") -> dict:
    """跑 A2A Coordinator，可選擇性注入 Podcast 觀點。

    Args:
        ticker: 股票代碼。
        podcast_context: format_podcast_context() 的輸出；空字串就是純 AI 分析。
    Returns:
        {"ticker", "podcast_enhanced", "agent_reports", "final_advice"}
    """
    query = f"分析 {ticker}"
    if podcast_context:
        query += f"\n\n{podcast_context}"

    runner = InMemoryRunner(app=coordinator_app)
    session = await runner.session_service.create_session(
        app_name="stockpulse_coordinator", user_id="podcast")

    reports, advice = {}, ""
    async for ev in runner.run_async(
        user_id="podcast", session_id=session.id,
        new_message=types.Content(role="user", parts=[types.Part(text=query)]),
    ):
        path = getattr(getattr(ev, "node_info", None), "path", None)
        name = path.split("/")[-1].split("@")[0] if path else None
        text = " ".join(p.text or "" for p in (ev.content.parts if ev.content else "")).strip()
        if text and name == "synthesiser":
            advice = text
        elif text and name:
            reports[name] = text

    return {"ticker": ticker, "podcast_enhanced": bool(podcast_context),
            "agent_reports": reports, "final_advice": advice}


print("✅ analyze_with_podcast() 已定義")

if RUN_PODCAST and HAS_KEY and podcast_ctx:
    result = await analyze_with_podcast(PODCAST_TICKERS[0], podcast_ctx)
    display(Markdown(f"## 🎙️ 融合 Podcast 觀點的分析（{result['ticker']}）\n\n"
                     f"{result['final_advice']}"))
else:
    print("⏭  沒有 Podcast context，跳過融合分析（§5 已經示範過純 AI 版本）")

# 9️⃣ 收尾：關掉背景 server

背景 thread 上的 uvicorn 是 daemon thread，kernel 結束會一起收掉，
但**在同一個 kernel 裡重跑整本之前最好手動關**，避免累積一堆閒置 server。

`srv.should_exit = True` 大約 1 秒內關完，port 立刻釋放，
重新執行 §5 的啟動 cell 可以直接重新綁定。

In [ ]:
stop_a2a()                                    # 關掉所有 A2A server

for key, info in list(API_SERVERS.items()):   # 關掉 FastAPI
    info["srv"].should_exit = True
    info["thread"].join(timeout=5)
    print(f"   🛑 FastAPI ({key}) 已關閉（thread alive={info['thread'].is_alive()}）")
    API_SERVERS.pop(key)

print("\n剩下的背景 thread:",
      [t.name for t in threading.enumerate()
       if t.name.startswith(("a2a-", "fastapi"))] or "（乾淨）")

# 🔟 Docker + Cloud Run 部署

## 專案結構

```
stockpulse-ai/
├── app/
│   ├── main.py                   # FastAPI app（§6 那段搬過來）
│   ├── data.py                   # §1 的資料層
│   ├── agents/
│   │   ├── market_analyst.py     # §3 的 Workflow
│   │   ├── trading_strategist.py # §4 的 LangGraph + 橋接
│   │   └── coordinator.py        # §5 的 Coordinator Workflow
│   └── routers/
├── frontend/                     # React（Vite + TS）
├── Dockerfile
└── pyproject.toml
```

## Dockerfile（multi-stage：Node build → Python runtime）

```dockerfile
# ---------- Stage 1: 前端 ----------
FROM node:22-slim AS frontend
WORKDIR /web
COPY frontend/package*.json ./
RUN npm ci
COPY frontend/ ./
RUN npm run build

# ---------- Stage 2: Python runtime ----------
FROM python:3.12-slim
WORKDIR /app

# graphviz 只有要在伺服器上產生 graph 圖片才需要
RUN apt-get update && apt-get install -y --no-install-recommends graphviz \
    && rm -rf /var/lib/apt/lists/*

# 用 uv 裝依賴（比 pip 快很多）
COPY --from=ghcr.io/astral-sh/uv:latest /uv /usr/local/bin/uv
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev

COPY app/ ./app/
COPY --from=frontend /web/dist ./static/

ENV PYTHONUNBUFFERED=1 PORT=8080
EXPOSE 8080
# Cloud Run 會注入 $PORT，一定要讀它
CMD ["sh", "-c", "uv run uvicorn app.main:app --host 0.0.0.0 --port ${PORT:-8080}"]
```

## 部署

```bash
export PROJECT_ID="your-project-id"
export REGION="asia-east1"
gcloud config set project $PROJECT_ID

# API Key 放 Secret Manager，不要用 --set-env-vars 明文塞
echo -n "$GOOGLE_API_KEY" | gcloud secrets create google-api-key --data-file=-

# 建置 + 部署
gcloud run deploy stockpulse-ai \
  --source . \
  --region $REGION \
  --allow-unauthenticated \
  --memory 2Gi \
  --cpu 2 \
  --timeout 600 \
  --concurrency 20 \
  --set-secrets GOOGLE_API_KEY=google-api-key:latest \
  --set-env-vars GEMINI_MODEL=gemini-2.5-flash

# 驗證
SERVICE_URL=$(gcloud run services describe stockpulse-ai --region $REGION --format='value(status.url)')
curl "$SERVICE_URL/api/health"
curl "$SERVICE_URL/a2a/market-analyst/.well-known/agent-card.json"
curl "$SERVICE_URL/api/analyze/NVDA"
```

## 部署時要注意的四件事

| 項目 | 為什麼 | 怎麼做 |
|---|---|---|
| **`--timeout`** | LLM 鏈路很慢，預設 300s 常常不夠 | 設 600s；更長的分析改用 Cloud Tasks 非同步 |
| **A2A 的 base URL** | AgentCard 上的 URL 決定 client 打哪裡；寫死 localhost 一上雲就壞 | 用環境變數 `A2A_BASE_URL`，本機是 `http://localhost:8080`，雲端是 service URL |
| **`--min-instances`** | 冷啟動要載 ADK + LangGraph，第一個請求會等很久 | 教學/demo 用 0 省錢；正式環境設 1 |
| **`InMemorySessionService`** | 存在記憶體，Cloud Run 一縮容就全沒了 | 正式環境換 `DatabaseSessionService`（Cloud SQL）或 `VertexAiSessionService` |

## 拆成多服務（真正的 A2A 佈署）

上面是「單一 service，A2A 走 localhost」的 Phase A。
要拆成多服務時，程式碼**只有 AgentCard 的 URL 要改**：

```bash
gcloud run deploy market-analyst      --source . --region $REGION   # :8080 只跑這個 agent
gcloud run deploy trading-strategist  --source . --region $REGION
gcloud run deploy coordinator --source . --region $REGION \
  --set-env-vars MARKET_ANALYST_CARD=https://market-analyst-xxx.run.app/.well-known/agent-card.json,\
TRADING_STRATEGIST_CARD=https://trading-strategist-xxx.run.app/.well-known/agent-card.json
```

```python
remote_market = RemoteA2aAgent(
    name="remote_market_analyst",
    agent_card=os.environ["MARKET_ANALYST_CARD"],   # ← 只有這一行不一樣
)
```

§5 的 Coordinator Workflow 整段不用動。**這就是 A2A 的價值。**

# 1️⃣1️⃣ ADK 1.x → 2.0 遷移對照表

如果你手上有 ADK 1.x 的教材或專案，這張表就是全部要改的東西。

## 編排

| ADK 1.x | ADK 2.0 | 說明 |
|---|---|---|
| `SequentialAgent(sub_agents=[a, b, c])` | `Workflow(edges=[(START, a), (a, b), (b, c)])` | 1.x 版本仍可用但會噴 DeprecationWarning |
| `ParallelAgent(sub_agents=[a, b])` | `Workflow(edges=[(START, (a, b))])` | 同上 |
| 兩者巢狀組合 | 一張 `Workflow` 的 DAG | 這才是換掉的真正原因 |
| ❌ 沒有 fan-in | `JoinNode` | **fan-in 目標一定要是 `JoinNode`**，普通 node 會被跑 N 次 |
| ❌ 沒有條件分支 | `(node, {"a": x, "b": y})` + `ctx.route = "a"` | 分支靠 `ctx.route`，**不是 return 值** |
| ❌ | `Workflow(max_concurrency=N)` | 限制同時執行的節點數 |
| ❌ | `node(fn, retry_config=…, timeout=…)` | 節點層級的重試與逾時 |

## 圖形

| ADK 1.x | ADK 2.0 |
|---|---|
| `google.adk.cli.utils.agent_graph.get_agent_graph()` | `graph_serialization.serialize_app_info()` + `graph_visualization.plot_workflow_graph()` |
| 只有靜態結構 | 可傳 `agent_state` 用執行狀態上色 |

`plot_workflow_graph(app_info, agent_state=None, format="svg", dark_mode=True)`
- `format="svg"` / `"dot"` 回傳 **str**；`"png"` / `"pdf"` / `"jpg"` 回傳 **bytes**
- `agent_state` 形狀：`{"nodes": {"節點名": {"status": 0-6}}}`
- `status` 必須是 **int 或 `NodeStatus` enum**；傳字串會被靜默當成 INACTIVE（白色）
- 節點名是**裸名**（`fetch_quote`），不是事件路徑（`wf@1/fetch_quote@1`）
- 要拿到狀態快照必須開 `App(resumability_config=ResumabilityConfig(is_resumable=True))`

## 執行

| ADK 1.x | ADK 2.0 |
|---|---|
| `Runner(agent=…, app_name=…, session_service=…)` | 同樣可用，但建議 `Runner(app=App(...), session_service=…)` |
| `InMemoryRunner(agent)` | `InMemoryRunner(agent=…)` 或 `InMemoryRunner(app=…)` 或 `InMemoryRunner(node=…)` |
| ❌ | `await runner.run_debug("問題", verbose=True)` — 一行跑起來 |
| ❌ | `Runner(auto_create_session=True)` — 不用先手動建 session |
| — | `App` 新增 `plugins` / `events_compaction_config` / `context_cache_config` / `resumability_config` |

## A2A

| ADK 1.x（a2a-sdk 0.3.x） | ADK 2.0（a2a-sdk 1.x） |
|---|---|
| `card["url"]` | **`card["supportedInterfaces"][0]["url"]`** |
| `card["protocolVersion"]` | `card["supportedInterfaces"][0]["protocolVersion"]` |
| `card["preferredTransport"]` | `card["supportedInterfaces"][0]["protocolBinding"]` |
| `method: "message/send"` | 兩種都支援：`message/send`（0.3 相容）或 `SendMessage`（1.x 原生，**要 `A2A-Version: 1.0` header**） |
| `state: "completed"` | 0.3 方言仍是 `completed`；1.x 方言是 `TASK_STATE_COMPLETED` |

## 其他

| 主題 | 重點 |
|---|---|
| `LangGraphAgent` | `LangGraphAgent(graph=compiled_graph, instruction=…)`；圖的 state 必須有 `messages` key，且最後一則訊息 content 必須是純字串 |
| `nest_asyncio` | **不要用**。Python 3.14 + uvicorn 0.52 會壞掉；ipykernel 7 原生支援 top-level `await` |
| `msg.content`（langchain） | Gemini 3.x 回傳 list of blocks，不是 str。用 `msg.text` |
| `yfinance` `.news` | 欄位在 `entry["content"]` 裡，不是平的 dict |
| `yfinance` `dividendYield` | 已經是百分比數字，不要再 `* 100` |

# 1️⃣2️⃣ 參考資料與檢核清單

## 官方文件

**ADK**
- ADK 文件：https://google.github.io/adk-docs/
- ADK Python GitHub：https://github.com/google/adk-python
- ADK Workflow（2.0 新編排模型）：https://google.github.io/adk-docs/agents/workflow-agents/
- ADK 範例：https://github.com/google/adk-samples

**A2A Protocol**
- A2A 規範：https://a2a-protocol.org/
- A2A GitHub：https://github.com/a2aproject/A2A
- a2a-python SDK：https://github.com/a2aproject/a2a-python

**Gemini / Google Cloud**
- Gemini API：https://ai.google.dev/gemini-api/docs
- AI Studio（拿 API Key）：https://aistudio.google.com/apikey
- google-genai SDK：https://googleapis.github.io/python-genai/
- Cloud Run：https://cloud.google.com/run/docs
- Vertex AI Agent Engine：https://cloud.google.com/vertex-ai/generative-ai/docs/agent-engine/overview

**LangGraph / FastAPI / yfinance**
- LangGraph：https://langchain-ai.github.io/langgraph/
- FastAPI：https://fastapi.tiangolo.com/
- yfinance：https://ranaroussi.github.io/yfinance/

---

## 這本 notebook 學到什麼

| # | 技能 | 在哪一節 | 怎麼驗收自己會了 |
|---|---|---|---|
| 1 | 把 Python 函式變成 ADK tool | §1 §2 | 能說出型別註解與 docstring 各自的作用 |
| 2 | `LlmAgent` + `Runner` + 事件流 | §2 | 能從事件流指出哪一步是 tool call、哪一步是最終回覆 |
| 3 | `Workflow` 的 DAG 語法 | §3 | 能徒手寫出 fan-out + JoinNode fan-in + 條件分支 |
| 4 | 為什麼 fan-in 一定要 `JoinNode` | §3 | 能解釋不用會發生什麼（節點跑 N 次） |
| 5 | 條件分支靠 `ctx.route` | §3 | 記得 return 值**不是** route |
| 6 | 把 graph 畫出來並上色 | §3 §5 | 能說出 `agent_state` 的形狀，以及為什麼要開 resumability |
| 7 | LangGraph 三節點流程 | §4 | 記得 `add_messages` 只回傳新訊息 |
| 8 | `LangGraphAgent` 橋接 | §4 | 能說出橋接對 graph state 的兩個要求 |
| 9 | `to_a2a()` 發佈 A2A 服務 | §5 | 能自己起一個 server 並抓到 AgentCard |
| 10 | 讀懂 AgentCard | §5 | 知道 1.x 的 endpoint 在 `supportedInterfaces` 裡 |
| 11 | 手寫 JSON-RPC 呼叫 A2A | §5 | 知道答案在 `result.artifacts[0].parts[0].text` |
| 12 | SSE streaming | §5 | 知道要先把 `capabilities.streaming` 打開 |
| 13 | `RemoteA2aAgent` 當 Workflow 節點 | §5 | 這是本教材的核心，能自己重寫一次 |
| 14 | FastAPI mount A2A app | §6 | 能在一個 service 裡同時提供 REST 和 A2A |
| 15 | Cloud Run 部署 | §10 | 知道 `$PORT`、Secret Manager、timeout 三個坑 |
| 16 | 1.x → 2.0 遷移 | §11 | 能把舊教材改過來 |

---

## 下一步練習（由淺到深）

1. **加一個 agent**：做一個「產業比較 agent」，把同產業的三支股票拉進來比較，
   接進 §5 的 Coordinator（提示：加一個 `RemoteA2aAgent` 節點 + 改 `merged` 的邊）
2. **加條件分支**：在 Coordinator 裡加一個 `decide` 節點，
   只有基本面評分 > 7 分才呼叫 Trading Strategist（省 token）
3. **換 session backend**：把 `InMemorySessionService` 換成
   `DatabaseSessionService`，讓對話可以跨重啟保留
4. **用 `LoopAgent`**：做一個「分析 → 自我批評 → 修正」的迴圈，最多跑 3 輪
5. **真的拆多服務**：把兩個 agent 各自部署到 Cloud Run，
   驗證 §5 的 Coordinator 只改 AgentCard URL 就能運作
6. **加 observability**：用 ADK 的 `plugins` 加一個記錄 token 用量的 plugin
7. **回測**：把 Trading Strategist 的訊號存起來，跑 3 個月回測算勝率

---

> ⚠️ **免責聲明**：本教材所有分析內容均由 AI 產生，僅供技術教學示範，
> **不構成任何投資建議**。實際投資請自行評估風險並諮詢專業人士。

**恭喜完成 StockPulse AI（ADK 2.0 版）！** 🎉